# 13 — Auditoria dos rótulos e dos erros prioritários

O experimento 12 aumentou a precisão em alguns pontos, sem recuperar os
requisitos de recall e de omissões de prioridade alta. Antes de outro
treinamento, vamos examinar os dados e os objetos em que isso ocorreu.

**Esta etapa funciona em CPU e não executa treinamento nem nova inferência.**
Os painéis dos modelos serão reconstruídos a partir dos CSVs já produzidos
pelo experimento 12. As 300 imagens reservadas continuam sem extração
e sem inferência. Home TEST e Open Images TEST também ficam fora da revisão.

| Grupo | Home | Open Images | Finalidade |
|---|---:|---:|---|
| Wardrobe em treino | 24 casos | 24 casos | Comparar guarda-roupa, closet, rack, aparador e estante |
| Mesas e cadeiras em treino | 12 casos | 12 casos | Revisar a classe e a caixa original |
| Erros e controles em validação | Até 12 casos | Até 24 casos | Examinar omissões, regressões, melhoras e acertos |

São até **108 casos**, cada um com uma caixa em foco. Uma imagem pode
aparecer em mais de um caso. A amostra é dirigida para auditoria e não
estima a frequência dos problemas na base inteira.


## 1. Executar no Kaggle

1. Importe este notebook e selecione **CPU / Accelerator: None**.
2. Ative Internet e habilite o Secret **GDRIVE_RCLONE_CONFIG_B64**.
3. Execute **Save Version → Save & Run All**, desde o começo.

Os caminhos dos experimentos anteriores estão preenchidos. O notebook baixa
Home, a validação congelada e somente os lotes de treino necessários dentre
os cinco já usados no piloto. Cada ZIP de Open Images é conferido e removido
do espaço temporário depois de recuperar os membros selecionados.

Ao terminar, baixe **pacote_auditoria_13.zip**. Dentro dele, **revisor_13.html**
é uma ficha visual que abre no navegador sem Internet. Você pode enviar o
pacote antes de preencher as fichas para analisarmos os casos por aqui.


In [ ]:
%pip install -q "numpy>=1.26" "pillow>=10" "scipy>=1.11" pyyaml


In [ ]:
import sys, json, hashlib, shutil, platform
from pathlib import Path
from datetime import datetime,timezone
from collections import Counter
import numpy as np
import PIL, scipy
from IPython.display import display,FileLink

SEED=42
NOME_EXPERIMENTO='auditoria13_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
PASTA_LOCAL=Path('/tmp/tcc_auditoria_13')/NOME_EXPERIMENTO
PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
PASTA_LOCAL.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
PASTA_EXECUCAO_10='resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010'
PASTA_EXECUCAO_11='resultados/diagnostico11_20261016_024603_683411'
PASTA_EXECUCAO_12='resultados/ajuste12_20261016_033819_596490'
# Opcional: JSON do revisor de uma execução anterior da MESMA amostra.
# Adicione-o via Add Input e preencha o caminho para consolidar as respostas.
# Na primeira execução, manter None; revisões exportadas podem ser importadas posteriormente.
ARQUIVO_REVISOES=None
print('Etapa de auditoria em CPU: zero novos treinos e zero novas inferências.')


In [ ]:
EXPECTED = {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'test_home': 'f48be0a2b3a93e59259b1ae0d3655028951bf270ef7f4e7916a9424530951a89', 'preparacao': '9865042654c51e83955c2aeb186a1b5e27e1028a5c56f1f163e2891abc0bec49', 'divisao_10': 'a5dd35ff484dda77e8988aefdc961f05d07ea409d1336cbe90c4a5420632bab9', 'manifesto_pacotes': '655094dea28755a7bddc817ae66660d06fab28ccb8f9d140d3ffebb07f03a0d4', 'pacote_11': 'a3f3fb500cef83fc4890f6bbec6725bbf5e080503fd0478af3dd84d6f5ef6e93', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'codigo_11': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2'}, 'pacote_12': '1c3a6016099c2bb23aca29396286497c7973f00136c1afbb0bbc62e45aa20650', 'codigo_12': {'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'diag11_data.py': 'c9f137d4dcc8c8f5fb156aaf25cddf6707d24e74567c05237332c2144e6e2f98', 'diag11_reports.py': '0068af8fd70b616cb642e9fb8a75d503cdc003f18fe79d64fd487bd7822d85e2', 'exp10_partial.py': '86a36f93aa643853f1268dcc133bed7e7344dea269b9d1d58cf54095538fffae', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'exp12_data.py': '0a9de7e99d302a2690f18f51924f4dc80b32861c9d2149ccfe0572ec75499f30', 'exp12_training.py': '6c51c9965336b2f00bd9b282e2159c0b63f07c5c16859da78da2818bd4202fdc', 'exp12_reports.py': '120609247939189f895c01cdfd1411e4d2b416d1e293cb26851d63d04835b49a'}}
LOCALIZACAO_11 = {'pasta_drive': 'datasets/validacao_independente_11_v1/695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'arquivo': 'validacao_independente_11_695b445e15c9.zip', 'arquivo_sha256': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'dataset_sha256': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'validacao': 1000, 'reserva_futura': 300}


## 2. Carregar o código embutido

Os extratores preservam os hashes e o mapeamento usados nos notebooks 11 e 12.
Não é necessário instalar Torch ou Ultralytics. Nenhum peso de modelo será
carregado. O revisor usa apenas os dados e os registros existentes.


In [ ]:
MODULOS_EMBUTIDOS = {'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'diag11_data.py': '"""Entradas, auditoria documental e nova validação, sem consulta a previsões."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport time\nimport urllib.request\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.fft import dctn\n\nfrom diag11_core import (ALL_KNOWN, ALERT_MASK, NAMES, canonical, parse_yolo,\n                         safe_member, save_csv, save_json, sha256_file)\n\nSOURCE_URLS = {\n    "classes": "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv",\n    "hierarquia": "https://storage.googleapis.com/openimages/2018_04/bbox_labels_600_hierarchy.json",\n    "caixas": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",\n    "rotulos": "https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels-boxable.csv",\n    "metadata": "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv",\n}\n\n\ndef download(url, path, retries=3):\n    """Download atômico, com recibo local e validação de tamanho quando informado."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    receipt = path.with_name(path.name + ".download.json")\n    if path.is_file() and receipt.is_file():\n        prior = json.loads(receipt.read_text())\n        if prior.get("url") == url and prior.get("sha256") == sha256_file(path):\n            return path\n    last = None\n    for attempt in range(retries):\n        tmp = path.with_name(path.name + ".tmp")\n        try:\n            req = urllib.request.Request(url, headers={"User-Agent": "TCC-Diagnostico-11/1.0"})\n            with urllib.request.urlopen(req, timeout=45) as response, tmp.open("wb") as out:\n                expected = response.headers.get("Content-Length")\n                for block in iter(lambda: response.read(2**20), b""):\n                    out.write(block)\n            if expected and tmp.stat().st_size != int(expected):\n                raise IOError("Tamanho recebido difere do Content-Length.")\n            if not tmp.stat().st_size:\n                raise IOError("Arquivo recebido vazio.")\n            tmp.replace(path)\n            save_json(receipt, {"url": url, "sha256": sha256_file(path),\n                                "bytes": path.stat().st_size})\n            return path\n        except Exception as exc:\n            last = type(exc).__name__ + ": " + str(exc)[:180]\n            tmp.unlink(missing_ok=True)\n            if attempt + 1 < retries:\n                time.sleep(min(2**attempt, 4))\n    raise IOError(f"Download não concluído: {url}; {last}")\n\n\ndef csv_rows(path):\n    with Path(path).open(encoding="utf-8-sig", newline="") as f:\n        yield from csv.DictReader(f)\n\n\ndef load_history(csv_path, json_path, expected_hash):\n    split = json.loads(Path(json_path).read_text())\n    stated = split.pop("divisao_sha256")\n    computed = hashlib.sha256(canonical(split).encode()).hexdigest()\n    if stated != expected_hash or computed != stated:\n        raise RuntimeError("Divisão histórica não corresponde à execução 10.")\n    rows = list(csv_rows(csv_path))\n    if len(rows) != 100000 or len({r["id"] for r in rows}) != 100000:\n        raise RuntimeError("O manifesto histórico precisa de 100.000 IDs distintos.")\n    if Counter(r["split"] for r in rows) != {"train": 80000, "test": 20000}:\n        raise RuntimeError("Divisão histórica diferente de 80/20.")\n    for label, key in [("train", "ids_treino"), ("test", "ids_teste")]:\n        if {r["id"] for r in rows if r["split"] == label} != set(split[key]):\n            raise RuntimeError("CSV e JSON históricos divergem.")\n    hashes = {r["imagem_sha256"] for r in rows}\n    if len(hashes) != len(rows):\n        raise RuntimeError("O histórico contém imagens repetidas em bytes.")\n    return {"ids": {r["id"] for r in rows}, "hashes": hashes, "rows": rows,\n            "split_hash": stated}\n\n\ndef phash(path):\n    with Image.open(path) as image:\n        small = np.asarray(image.convert("L").resize((32, 32), Image.Resampling.LANCZOS),\n                           dtype=float)\n    low = dctn(small, norm="ortho")[:8, :8]\n    bits = low > np.median(low.flatten()[1:])\n    bits[0, 0] = False\n    return int.from_bytes(np.packbits(bits.flatten()).tobytes(), "big")\n\n\nclass NearIndex:\n    """pHash de 64 bits, distância <=4; sinal de semelhança, não prova semântica."""\n    def __init__(self):\n        self.tables = [defaultdict(list) for _ in range(8)]\n\n    def add(self, value, name):\n        for band in range(8):\n            self.tables[band][(value >> (8*band)) & 255].append((value, name))\n\n    def find(self, value, max_distance=4):\n        checked = set()\n        for band in range(8):\n            for prior, name in self.tables[band].get((value >> (8*band)) & 255, []):\n                if (prior, name) in checked:\n                    continue\n                checked.add((prior, name))\n                if (prior ^ value).bit_count() <= max_distance:\n                    return name\n        return None\n\n\ndef extract_home(zip_path, destination, expected_zip, expected_test, expected_policy):\n    if sha256_file(zip_path) != expected_zip:\n        raise RuntimeError("ZIP Home diferente da entrada preservada.")\n    destination = Path(destination)\n    records, hashes, near = [], set(), NearIndex()\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Há membros repetidos no ZIP Home.")\n        manifest = json.loads(z.read("verificacao_preparacao.json"))\n        if (manifest["test"]["sha256"] != expected_test or\n                manifest["test"]["imagens"] != 403 or\n                {int(k): v for k, v in manifest["classes"].items()} != dict(enumerate(NAMES))):\n            raise RuntimeError("Manifesto Home incompatível.")\n        root = manifest["dataset_dir"]\n        items = {r["arquivo"]: r for r in manifest["arquivos"]}\n        if len(items) != len(manifest["arquivos"]):\n            raise RuntimeError("Manifesto Home com membros repetidos.")\n        for name in z.namelist():\n            safe_member(name)\n            if name.startswith("codigo/") and name.endswith(".py"):\n                raw = z.read(name)\n                if name in items and hashlib.sha256(raw).hexdigest() != items[name]["sha256"]:\n                    raise RuntimeError("Código Home diferente do manifesto.")\n                out = destination / name\n                out.parent.mkdir(parents=True, exist_ok=True)\n                out.write_bytes(raw)\n        for name, item in sorted(items.items()):\n            is_image = (name.startswith(f"{root}/images/") and\n                        Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"})\n            if is_image:\n                hashes.add(item["sha256"])\n            role = None\n            if name.startswith(f"{root}/images/train/raw_train/"):\n                role = "home_train"\n            elif name.startswith(f"{root}/images/val/raw_train/"):\n                role = "home_val"\n            if not is_image or role is None:\n                continue\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise RuntimeError(f"Imagem Home alterada: {name}")\n            image = destination / name\n            image.parent.mkdir(parents=True, exist_ok=True)\n            image.write_bytes(raw)\n            label_name = name.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"\n            if label_name not in items:\n                raise RuntimeError("Label Home ausente do manifesto.")\n            label_raw = z.read(label_name)\n            if hashlib.sha256(label_raw).hexdigest() != items[label_name]["sha256"]:\n                raise RuntimeError("Label Home alterado.")\n            with Image.open(image) as im:\n                width, height = im.size\n                im.verify()\n            gt = parse_yolo(label_raw, width, height, source=label_name)\n            record = {"id": "home:" + name, "name": name.split("/raw_train/")[-1],\n                      "image": image, "role": role, "coorte": "home_val_historica",\n                      "width": width, "height": height, "known_mask": ALL_KNOWN,\n                      "sha256": item["sha256"], "label_sha256": items[label_name]["sha256"],\n                      "gt": gt}\n            record["phash"] = phash(image)\n            near.add(record["phash"], name)\n            records.append(record)\n    counts = Counter(r["role"] for r in records)\n    if counts != {"home_train": 1824, "home_val": 456}:\n        raise RuntimeError(f"Home inesperado: {dict(counts)}")\n    train_hash = {r["sha256"] for r in records if r["role"] == "home_train"}\n    val_hash = {r["sha256"] for r in records if r["role"] == "home_val"}\n    if train_hash & val_hash:\n        raise RuntimeError("Home TRAIN e VAL atuais compartilham imagens em bytes.")\n    policy_file = destination / "codigo/src/politica_alertas.py"\n    if not policy_file.is_file() or sha256_file(policy_file) != expected_policy:\n        raise RuntimeError("Política V2 diferente da versão preservada.")\n    audit = {"home_train_atual": 1824, "home_val_atual": 456,\n             "test_home_preservado": 403, "test_extraido": False,\n             "intersecao_bytes_home_train_val_atual": 0,\n             "divergencia_documental_1822_1824": "Não reconciliada; o pacote atual tem 1824.",\n             "separacao_atual_nao_comprova_historico_baseline": True}\n    return records, hashes, near, audit\n\n\ndef audit_baseline_metadata(checkpoint, expected_hash, home_records, evidence=None):\n    if sha256_file(checkpoint) != expected_hash:\n        raise RuntimeError("Peso baseline diferente da referência.")\n    # Só o checkpoint autenticado pelo SHA esperado é desserializado.\n    import torch\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=False)\n    args = obj.get("train_args", {})\n    fields = ["data", "epochs", "batch", "imgsz", "seed", "optimizer", "lr0",\n              "project", "name", "fraction"]\n    audit = {"modelo_sha256": expected_hash, "data_checkpoint": str(obj.get("date")),\n             "train_args_preservados": {k: args[k] for k in fields if k in args},\n             "estado_proveniencia_original": "nao_determinado",\n             "home_val_usada_no_treino_original": None,\n             "motivo": "Caminhos e argumentos não contêm a lista de imagens vistas no treino original."}\n    del obj\n    if evidence is not None:\n        required = {"modelo_sha256", "train_image_sha256", "val_image_sha256"}\n        if not required <= evidence.keys() or evidence["modelo_sha256"] != expected_hash:\n            audit["evidencia_original"] = "incompativel_ou_sem_vinculo_ao_peso"\n        else:\n            train = set(evidence["train_image_sha256"])\n            val = set(evidence["val_image_sha256"])\n            valid_sha = lambda x: isinstance(x, str) and len(x) == 64 and all(c in "0123456789abcdef" for c in x)\n            if not train or any(not valid_sha(h) for h in train | val):\n                raise ValueError("Manifesto original precisa de hashes SHA-256 válidos.")\n            current = {r["sha256"] for r in home_records if r["role"] == "home_val"}\n            overlap = train & current\n            complete = evidence.get("manifesto_treino_completo") is True\n            audit.update(estado_proveniencia_original="manifesto_documental_analisado",\n                         evidencia_original="documento_vinculado_ao_sha_do_modelo",\n                         home_val_usada_no_treino_original=True if overlap else (False if complete else None),\n                         manifesto_declarado_completo=complete,\n                         imagens_home_val_no_manifesto_train_original=len(overlap),\n                         imagens_home_val_no_manifesto_val_original=len(current & val),\n                         motivo="Conclusão limitada à completude e correção do manifesto apresentado.")\n    return audit\n\n\ndef stable_order(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef balanced_ids(records, n, seed=42, per_class=45):\n    """Diversidade por classe e preenchimento determinístico, sem usar previsões."""\n    if len(records) < n:\n        raise ValueError(f"Há somente {len(records)} imagens elegíveis; exigidas {n}.")\n    groups = {c: sorted((iid for iid, r in records.items() if\n                         r["positive_mask"] & (1 << c)), key=lambda x: stable_order(x, seed))\n              for c in range(12)}\n    order = sorted(groups, key=lambda c: (len(groups[c]), c))\n    chosen, counts = set(), Counter()\n    # Diversificar exemplos negativos quando as seis classes têm ausência verificada.\n    negatives = sorted((iid for iid, r in records.items() if\n                        r["known_mask"] & ALERT_MASK == ALERT_MASK and\n                        not r["positive_mask"] & ALERT_MASK), key=lambda x: stable_order(x, seed))\n    for iid in negatives[:min(len(negatives), n//10)]:\n        chosen.add(iid)\n        counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n    for target in range(1, per_class+1):\n        for cid in order:\n            if len(chosen) >= n:\n                break\n            if counts[cid] >= target:\n                continue\n            for iid in groups[cid]:\n                if iid not in chosen:\n                    chosen.add(iid)\n                    counts.update(c for c in range(12) if records[iid]["positive_mask"] & (1 << c))\n                    break\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return sorted(chosen, key=lambda x: stable_order(x, seed))\n\n\ndef split_pool(records, n_reserve, seed=42):\n    """Estratos pela classe positiva mais rara; a reserva não entra na inferência."""\n    frequencies = Counter(c for r in records.values() for c in range(12)\n                          if r["positive_mask"] & (1 << c))\n    groups = defaultdict(list)\n    for iid, record in records.items():\n        classes = record["classes_positivas_avaliaveis"]\n        primary = min(classes, key=lambda c: (frequencies[c], c)) if classes else -1\n        groups[primary].append(iid)\n    total = len(records)\n    if not 0 < n_reserve < total:\n        raise ValueError("A reserva precisa ser menor que o conjunto novo.")\n    quotas = {c: len(ids)*n_reserve//total for c, ids in groups.items()}\n    order = sorted(groups, key=lambda c: (-(len(groups[c])*n_reserve % total), c))\n    for cid in order[:n_reserve-sum(quotas.values())]:\n        quotas[cid] += 1\n    selected = set()\n    for cid, ids in groups.items():\n        selected.update(sorted(ids, key=lambda x: stable_order(x, seed+1))[:quotas[cid]])\n    if len(selected) != n_reserve:\n        raise RuntimeError("A divisão não produziu a reserva planejada.")\n    return {iid: ("reserva_futura" if iid in selected else "val_nova") for iid in records}\n\n\ndef filter_metadata(records, metadata_path, prior_ids):\n    metadata, exclusions = {}, Counter()\n    for row in csv_rows(metadata_path):\n        iid = row["ImageID"]\n        if iid not in records:\n            continue\n        if row["Subset"] != "validation":\n            raise ValueError("A metadata contém uma origem diferente de validation.")\n        if iid in prior_ids:\n            exclusions["id_presente_nos_100k"] += 1\n            continue\n        try:\n            rotation = float(row["Rotation"])\n        except (TypeError, ValueError):\n            rotation = float("nan")\n        # Evita suposições sobre transformação das caixas e orientação do JPEG.\n        if rotation != 0:\n            exclusions["rotacao_nao_zero_ou_desconhecida"] += 1\n            continue\n        metadata[iid] = {k: row.get(k, "") for k in\n                         ["ImageID", "OriginalLandingURL", "License", "Author", "Rotation"]}\n    result = {iid: {**r, "creditos": metadata[iid]} for iid, r in records.items()\n              if iid in metadata}\n    return result, dict(exclusions)\n\n\ndef download_images(candidates, ids, directory, workers=8):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    def one(iid):\n        record = dict(candidates[iid])\n        path = directory / f"oi_val_{iid}.jpg"\n        try:\n            download(f"https://open-images-dataset.s3.amazonaws.com/validation/{iid}.jpg", path)\n            with Image.open(path) as image:\n                image.verify()\n            with Image.open(path) as image:\n                width, height = image.size\n                orientation = image.getexif().get(274, 1)\n                if width < 32 or height < 32 or orientation != 1:\n                    raise ValueError("JPEG pequeno ou com orientação EXIF não trivial.")\n                image.load()\n            tiny = {g["classe_id"] for g in record["gt_normalizado"] if\n                    (g["bbox_norm"][2]-g["bbox_norm"][0])*width < 1 or\n                    (g["bbox_norm"][3]-g["bbox_norm"][1])*height < 1}\n            # A avaliação da ausência fica desconhecida para a classe afetada.\n            record["restricoes_cobertura"] = dict(record["restricoes_cobertura"])\n            for cid in tiny:\n                for key in ["known_mask", "positive_mask", "negative_mask"]:\n                    record[key] &= ~(1 << cid)\n                record["restricoes_cobertura"][str(cid)] = ["caixa_menor_que_um_pixel"]\n            record["classes_positivas_avaliaveis"] = [c for c in range(12)\n                                                      if record["positive_mask"] & (1 << c)]\n            if not (record["positive_mask"] or record["known_mask"] & ALERT_MASK == ALERT_MASK):\n                raise ValueError("Nenhuma cobertura útil após conferir as dimensões reais.")\n            record.update(image=path, width=width, height=height, sha256=sha256_file(path),\n                          phash=phash(path), name=path.name)\n            return record, None\n        except Exception as exc:\n            return None, {"id": iid, "motivo": type(exc).__name__, "detalhe": str(exc)[:240]}\n\n    valid, failures = {}, []\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for done, (record, failure) in enumerate(executor.map(one, ids), 1):\n            if record:\n                valid[record["id"]] = record\n            else:\n                failures.append(failure)\n            if done % 100 == 0 or done == len(ids):\n                print(f"Novas imagens: {done}/{len(ids)}; válidas: {len(valid)}", flush=True)\n    return valid, failures\n\n\ndef deduplicate(records, prior_hashes, home_near, seed=42):\n    accepted, excluded, seen_hash, near = {}, [], set(), NearIndex()\n    for iid in sorted(records, key=lambda x: stable_order(x, seed)):\n        record = records[iid]\n        reason, similar = None, None\n        if record["sha256"] in prior_hashes:\n            reason = "bytes_presentes_nos_dados_anteriores"\n        elif record["sha256"] in seen_hash:\n            reason = "bytes_repetidos_no_conjunto_novo"\n        elif (similar := home_near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_com_home_train_ou_val"\n        elif (similar := near.find(record["phash"])) is not None:\n            reason = "semelhanca_phash_dentro_do_conjunto_novo"\n        if reason:\n            excluded.append({"id": iid, "motivo": reason, "semelhante_a": similar})\n            continue\n        accepted[iid] = record\n        seen_hash.add(record["sha256"])\n        near.add(record["phash"], iid)\n    return accepted, excluded\n\n\ndef freeze_dataset(records, roles, directory, source_hashes, prior_split_hash, seed=42):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    manifest, runtime = [], []\n    for iid in sorted(roles):\n        r, role = records[iid], roles[iid]\n        boxes = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            boxes.append({**g, "bbox": [x1*r["width"], y1*r["height"],\n                                       x2*r["width"], y2*r["height"]]})\n        coverage = {k: r[k] for k in ["known_mask", "positive_mask", "negative_mask",\n                                      "restricoes_cobertura"]}\n        label = directory / "labels" / role / f"oi_val_{iid}.txt"\n        coverage_file = directory / "coverage" / role / f"oi_val_{iid}.json"\n        label.parent.mkdir(parents=True, exist_ok=True)\n        lines = []\n        for g in r["gt_normalizado"]:\n            x1, y1, x2, y2 = g["bbox_norm"]\n            lines.append(f"{g[\'classe_id\']} {(x1+x2)/2:.10f} {(y1+y2)/2:.10f} {(x2-x1):.10f} {(y2-y1):.10f}")\n        label.write_text("\\n".join(lines)+("\\n" if lines else ""), encoding="utf-8")\n        save_json(coverage_file, coverage)\n        image_rel = f"images/{role}/oi_val_{iid}.jpg"\n        item = {"id": iid, "role": role, "image": image_rel,\n                "label": label.relative_to(directory).as_posix(),\n                "coverage": coverage_file.relative_to(directory).as_posix(),\n                "sha256": r["sha256"], "label_sha256": sha256_file(label),\n                "coverage_sha256": sha256_file(coverage_file),\n                "width": r["width"], "height": r["height"], "known_mask": r["known_mask"],\n                "positive_mask": r["positive_mask"], "negative_mask": r["negative_mask"],\n                "phash": f"{r[\'phash\']:016x}", "creditos": r["creditos"],\n                "gt_normalizado": r["gt_normalizado"]}\n        manifest.append(item)\n        runtime.append({**r, "role": role, "coorte": "openimages_val_nova",\n                        "gt": boxes, "label": label, "coverage": coverage_file})\n    metadata = {"versao": "11-validacao-independente-v1", "seed": seed, "nomes": NAMES,\n                "fontes_metadata_sha256": source_hashes, "origem_split": "validation",\n                "divisao_100k_anterior_sha256": prior_split_hash,\n                "contagens": dict(Counter(roles.values())),\n                "criterio": "Diversidade de classes, hash determinístico e pHash; sem previsões",\n                "cobertura": "Positivo explícito na raiz ou negativo na raiz/ancestral; classes problemáticas excluídas",\n                "independencia_limitada_ao_historico_conhecido": True,\n                "pretreinamentos_publicos_nao_auditados": True,\n                "reserva_futura_avaliada": False, "arquivos": manifest}\n    dataset_hash = hashlib.sha256(canonical(metadata).encode()).hexdigest()\n    metadata["dataset_sha256"] = dataset_hash\n    save_json(directory / "manifesto_validacao_nova.json", metadata)\n    save_csv(directory / "divisao.csv", [{k: r[k] for k in\n                                        ["id", "role", "sha256", "known_mask", "positive_mask"]}\n                                       for r in manifest])\n    archive = directory.parent / f"validacao_independente_11_{dataset_hash[:12]}.zip"\n    # Os JPEGs permanecem sem recompressão.\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:\n        z.write(directory / "manifesto_validacao_nova.json", "manifesto_validacao_nova.json")\n        z.write(directory / "divisao.csv", "divisao.csv")\n        for r, item in zip(sorted(runtime, key=lambda r: r["id"]), manifest):\n            z.write(r["image"], item["image"])\n            z.write(r["label"], item["label"])\n            z.write(r["coverage"], item["coverage"])\n    return runtime, metadata, archive\n\n\ndef support_rows(records):\n    rows = []\n    for role in sorted({r["role"] for r in records}):\n        group = [r for r in records if r["role"] == role]\n        for cid, name in enumerate(NAMES):\n            gt = [g for r in group for g in r["gt"] if\n                  g["classe_id"] == cid and r["known_mask"] & (1 << cid)]\n            rows.append({"conjunto": role, "classe_id": cid, "classe": name,\n                         "imagens_cobertura_conhecida": sum(bool(r["known_mask"] & (1 << cid)) for r in group),\n                         "imagens_positivas_avaliaveis": sum(bool(r["positive_mask"] & (1 << cid)) for r in group),\n                         "caixas_avaliaveis": len(gt), "suporte_reduzido": len(gt) < 30})\n    return rows\n', 'diag11_reports.py': '"""Comparação pareada, indícios dos erros e galeria de inspeção."""\nfrom __future__ import annotations\n\nimport gzip\nimport html\nimport json\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFont\n\nfrom diag11_core import (NAMES, LEVEL, alert_on, canonical, ratio, save_csv,\n                         save_json, with_policy)\nfrom diag11_data import stable_order\n\n\ndef paired_bootstrap(old_images, new_images, repetitions=1000, seed=42):\n    old = {r["id"]: r for r in old_images}\n    new = {r["id"]: r for r in new_images}\n    if set(old) != set(new) or len(old) != len(old_images) or len(new) != len(new_images):\n        raise ValueError("Bootstrap exige as mesmas imagens, sem repetições.")\n    ids = sorted(old)\n    fields = ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta", "alta_ref"]\n    a = np.array([[old[i][k] for k in fields] for i in ids], dtype=float)\n    b = np.array([[new[i][k] for k in fields] for i in ids], dtype=float)\n\n    def scores(x):\n        tp, fn, fp, missing, high = x\n        return [100*ratio(tp, tp+fp) if tp+fp else np.nan,\n                100*ratio(tp, tp+fn) if tp+fn else np.nan,\n                100*ratio(2*tp, 2*tp+fp+fn) if 2*tp+fp+fn else np.nan,\n                fn, fp, missing, 100*ratio(missing, high) if high else np.nan]\n\n    point = np.array(scores(b.sum(axis=0))) - np.array(scores(a.sum(axis=0)))\n    rng = np.random.default_rng(seed)\n    results = []\n    for _ in range(repetitions):\n        index = rng.integers(0, len(ids), len(ids))\n        results.append(np.array(scores(b[index].sum(axis=0))) -\n                       np.array(scores(a[index].sum(axis=0))))\n    samples = np.asarray(results)\n    labels = ["precision_alertas_pp", "recall_alertas_pp", "f1_alertas_pp",\n              "fn_alertas", "fp_alertas", "alta_sem_alerta", "taxa_alta_sem_alerta_pp"]\n    rows = []\n    for j, name in enumerate(labels):\n        values = samples[np.isfinite(samples[:, j]), j]\n        low, high = np.quantile(values, [.025, .975]) if len(values) else (None, None)\n        rows.append({"metrica": name, "delta_candidato_menos_antigo": float(point[j]) if np.isfinite(point[j]) else None,\n                     "ic95_inferior": low, "ic95_superior": high,\n                     "reamostragens_validas": len(values), "imagens": len(ids),\n                     "metodo": "Bootstrap pareado por imagem; não corrige possível exposição prévia"})\n    return rows\n\n\ndef comparison_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for name, result in models.items():\n            s, a = result["summary"], result["summary"]["alertas_por_objeto"]\n            rows.append({"coorte": cohort, "modelo": name, "imagens": s["imagens"],\n                         "precision_alertas": a["precision"], "recall_alertas": a["recall"],\n                         "f1_alertas": a["f1"], "tp_alerta": a["tp"], "fn_alerta": a["fn"],\n                         "fp_alerta": a["fp"], "alta_ref": s["alta_ref"],\n                         "alta_sem_alerta": s["alta_sem_alerta"],\n                         "map50_recorte": s["map50_recorte"],\n                         "map50_95_recorte": s["map50_95_recorte"]})\n    return rows\n\n\ndef diagnosis_rows(evaluations):\n    rows = []\n    for cohort, models in evaluations.items():\n        for model, result in models.items():\n            events = result["events"]\n            for cid, name in enumerate(NAMES):\n                group = [e for e in events if e["classe_id"] == cid]\n                for indication in sorted({x for e in group for x in e["indicios"].split("|") if x}):\n                    part = [e for e in group if indication in e["indicios"].split("|")]\n                    rows.append({"coorte": cohort, "modelo": model, "classe": name,\n                                 "indicio": indication, "eventos": len(part),\n                                 "fn_alerta": sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                 "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part),\n                                 "observacao": "Indícios podem se sobrepor; não estabelecem a causa do erro."})\n    return rows\n\n\ndef load_prediction_cache(path):\n    records = {}\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        for line in f:\n            r = json.loads(line)\n            if r["id"] in records:\n                raise ValueError("Cache tem imagem repetida.")\n            records[r["id"]] = r["predicoes"]\n    return records\n\n\ndef select_gallery_cases(old, new, limit=24, seed=42):\n    a, b = {r["id"]: r for r in old}, {r["id"]: r for r in new}\n    if set(a) != set(b):\n        raise ValueError("Galeria exige os dois modelos nas mesmas imagens.")\n    chosen = {}\n\n    def take(label, ordered, number):\n        used = 0\n        for iid in ordered:\n            if iid not in chosen and len(chosen) < limit:\n                chosen[iid] = label\n                used += 1\n                if used == number:\n                    break\n\n    worse = sorted(a, key=lambda i: (-(b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"]),\n                                     -(b[i]["fn_alerta"]-a[i]["fn_alerta"]), stable_order(i, seed)))\n    better = sorted(a, key=lambda i: (b[i]["alta_sem_alerta"]-a[i]["alta_sem_alerta"],\n                                      b[i]["fn_alerta"]-a[i]["fn_alerta"], stable_order(i, seed)))\n    false = sorted(a, key=lambda i: (-(a[i]["fp_alerta"]+b[i]["fp_alerta"]), stable_order(i, seed)))\n    take("Candidato com mais omissões", [i for i in worse if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) >\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Candidato com menos omissões", [i for i in better if\n         (b[i]["alta_sem_alerta"], b[i]["fn_alerta"]) <\n         (a[i]["alta_sem_alerta"], a[i]["fn_alerta"])], limit//4)\n    take("Falsos alertas avaliáveis", [i for i in false if a[i]["fp_alerta"]+b[i]["fp_alerta"]], limit//4)\n    take("Amostra por hash, independente do erro", sorted(a, key=lambda i: stable_order(i, seed)), limit)\n    return [{"id": i, "categoria": label} for i, label in chosen.items()]\n\n\ndef font(size):\n    candidates = ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",\n                  "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"]\n    for path in candidates:\n        if Path(path).is_file():\n            return ImageFont.truetype(path, size)\n    return ImageFont.load_default()\n\n\ndef draw_panel(record, objects, title):\n    canvas = Image.new("RGB", (540, 470), "#101a28")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 10), title, fill="#f5f7fa", font=font(18))\n    with Image.open(record["image"]) as image:\n        image = image.convert("RGB")\n        scale = min(520/image.width, 390/image.height)\n        image = image.resize((max(1, round(image.width*scale)),\n                              max(1, round(image.height*scale))), Image.Resampling.LANCZOS)\n    x0, y0 = (540-image.width)//2, 46+(390-image.height)//2\n    canvas.paste(image, (x0, y0))\n    sx, sy = image.width/record["width"], image.height/record["height"]\n    colours = {"alto": "#ff7b8a", "medio": "#ffde70",\n               "baixo": "#6ee7b7", "sem_prioridade": "#adbac7"}\n    for obj in objects:\n        cid = obj["classe_id"]\n        known = bool(record["known_mask"] & (1 << cid))\n        colour = colours[obj["prioridade"]] if known else "#a9afba"\n        x1, y1, x2, y2 = obj["bbox"]\n        box = [x0+x1*sx, y0+y1*sy, x0+x2*sx, y0+y2*sy]\n        draw.rectangle(box, outline=colour, width=2)\n        label = obj["classe"]\n        if "confianca" in obj:\n            label += f" {obj[\'confianca\']:.2f}"\n        label += " / " + obj["prioridade"] if known else " / sem cobertura"\n        # Limita o rótulo à largura visível; não altera a caixa nem as métricas.\n        tx, ty = max(0, min(box[0], 535)), max(44, box[1]-15)\n        text_box = draw.textbbox((tx, ty), label, font=font(11))\n        draw.rectangle(text_box, fill="#101a28")\n        draw.text((tx, ty), label, fill=colour, font=font(11))\n    draw.text((12, 445), "Rosa: alta | amarelo: média | verde: baixa | cinza: sem referência",\n              fill="#c6ced9", font=font(10))\n    return canvas\n\n\ndef render_case(record, old_pred, new_pred, policy, destination):\n    gt = with_policy([g for g in record["gt"] if record["known_mask"] & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    old = with_policy([p for p in old_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    new = with_policy([p for p in new_pred if p["confianca"] >= .25],\n                      record["width"], record["height"], policy)\n    panels = [draw_panel(record, gt, "Referência V2 avaliável"),\n              draw_panel(record, old, "Modelo antigo / confiança 0,25"),\n              draw_panel(record, new, "Candidato / confiança 0,25")]\n    result = Image.new("RGB", (1620, 470), "#101a28")\n    for i, panel in enumerate(panels):\n        result.paste(panel, (540*i, 0))\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    result.save(destination, quality=88)\n    return destination\n\n\ndef format_percent(value):\n    return "sem suporte" if value is None else f"{100*value:.2f}%"\n\n\ndef build_report(output_dir, evaluations, cohorts, audit, bootstrap, policy, dataset_manifest, seed=42):\n    output_dir = Path(output_dir)\n    comparison = comparison_rows(evaluations)\n    diagnosis = diagnosis_rows(evaluations)\n    save_csv(output_dir / "comparacao_modelos.csv", comparison)\n    save_csv(output_dir / "diagnostico_erros.csv", diagnosis)\n    save_csv(output_dir / "bootstrap_pareado.csv", bootstrap)\n    gallery = []\n    for cohort, models in evaluations.items():\n        records = {r["id"]: r for r in cohorts[cohort]}\n        caches = {name: load_prediction_cache(result["cache_file"]) for name, result in models.items()}\n        cases = select_gallery_cases(models["antigo"]["images"], models["candidato"]["images"], seed=seed)\n        old_images = {r["id"]: r for r in models["antigo"]["images"]}\n        new_images = {r["id"]: r for r in models["candidato"]["images"]}\n        for i, case in enumerate(cases):\n            record = records[case["id"]]\n            relative = f"galeria/{cohort}_{i:03}.jpg"\n            render_case(record, caches["antigo"][record["id"]],\n                        caches["candidato"][record["id"]], policy, output_dir/relative)\n            gallery.append({**case, "coorte": cohort, "imagem": record["name"], "arquivo": relative,\n                            "antigo": {k: old_images[record["id"]][k] for k in\n                                       ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "candidato": {k: new_images[record["id"]][k] for k in\n                                          ["tp_alerta", "fn_alerta", "fp_alerta", "alta_sem_alerta"]},\n                            "creditos": record.get("creditos", {})})\n    save_json(output_dir / "indice_galeria.json", gallery)\n    new_records = cohorts["openimages_val_nova"]\n    save_csv(output_dir / "revisao_cenarios.csv", [\n        {"id": r["id"], "imagem": r["name"], "ambiente_interno": "nao_revisado",\n         "proximo_do_uso_pretendido": "nao_revisado", "anotacao_parece_correta": "nao_revisado",\n         "observacao": "", "origem": r.get("creditos", {}).get("OriginalLandingURL", "")}\n        for r in new_records])\n    provenance = audit["baseline_original"]["estado_proveniencia_original"]\n    table = "<table><thead><tr><th>Conjunto</th><th>Modelo</th><th>Precisão alertas</th><th>Recall alertas</th><th>F1</th><th>FN</th><th>Alta sem alerta</th></tr></thead><tbody>"\n    for r in comparison:\n        table += "<tr>" + "".join("<td>"+html.escape(str(x))+"</td>" for x in\n                     [r["coorte"], r["modelo"], format_percent(r["precision_alertas"]),\n                      format_percent(r["recall_alertas"]), format_percent(r["f1_alertas"]),\n                      r["fn_alerta"], r["alta_sem_alerta"]]) + "</tr>"\n    table += "</tbody></table>"\n    cards = []\n    for case in gallery:\n        credit = case["creditos"]\n        source_url = str(credit.get("OriginalLandingURL", ""))\n        license_url = str(credit.get("License", ""))\n        attribution = html.escape(str(credit.get("Author", "")))\n        if source_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(source_url, quote=True)+\'">Fonte original</a>\'\n        if license_url.startswith(("https://", "http://")):\n            attribution += \' · <a href="\'+html.escape(license_url, quote=True)+\'">Licença</a>\'\n        details = (f"Antigo: FN {case[\'antigo\'][\'fn_alerta\']}, FP {case[\'antigo\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'antigo\'][\'alta_sem_alerta\']}. "\n                   f"Candidato: FN {case[\'candidato\'][\'fn_alerta\']}, FP {case[\'candidato\'][\'fp_alerta\']}, "\n                   f"alta sem alerta {case[\'candidato\'][\'alta_sem_alerta\']}.")\n        cards.append("<article><h3>"+html.escape(case["coorte"]+" / "+case["categoria"])+\n                     "</h3><p>"+html.escape(case["imagem"]+" — "+details)+"</p>"+\n                     \'<img loading="lazy" src="\'+case["arquivo"]+\'" alt="Caixas de referência e dos dois modelos">\'+\n                     "<p class=small>"+attribution+\n                     " · Sobreposição de caixas para avaliação.</p></article>")\n    html_report = """<!doctype html><html lang="pt-BR"><meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>TCC — Diagnóstico dos detectores</title><style>\nbody{font:16px/1.6 system-ui,sans-serif;background:#f3f5f7;color:#142135;margin:0}\nmain{max-width:1400px;margin:auto;padding:32px}h1{font-size:32px;line-height:1.2}\n.note,article,table{background:white;border:1px solid #dbe1e8;border-radius:12px}\n.note,article{padding:20px;margin:20px 0}table{border-collapse:collapse;width:100%}\ntd,th{padding:10px;text-align:left;border-bottom:1px solid #e6eaf0}th{background:#e8eff8}\nimg{max-width:100%;height:auto}.small{font-size:12px;color:#526174}.scroll{overflow:auto}\n</style><main><h1>Diagnóstico comparativo dos detectores</h1>\n<p>Pesos congelados · 12 classes · política V2 · confiança 0,25 · sem treinamento</p>\n<div class=note><strong>Como interpretar</strong><p>\nA Home VAL é histórica e exploratória. A validação externa é nova em relação aos IDs\ne bytes dos dados conhecidos do projeto; pré-treinamentos públicos não foram auditados.\nO recorte Open Images tem cobertura parcial e não garante cenas internas.\nAs métricas de alertas medem concordância com a V2, não risco físico independente.\nA reserva futura não foi inferida. Nenhum modelo foi promovido automaticamente.</p>\n<p>Proveniência documental do baseline: <strong>""" + html.escape(provenance) + """</strong>.</p></div>\n<h2>Alertas por objeto avaliável</h2><div class=scroll>""" + table + """</div>\n<p>Os CSVs trazem suporte por classe, AP no recorte, contagens e intervalos pareados.\nNão compare APs de protocolos e distribuições diferentes como se fossem o mesmo benchmark.</p>\n<h2>Casos para inspeção</h2><p>Os casos abaixo priorizam diferenças e erros, além de uma amostra por hash.\nEsta galeria não é uma estimativa da frequência dos erros. As contagens acima usam todas as imagens.\nObjetos sem cobertura aparecem em cinza e não são classificados automaticamente como falsos positivos.</p>\n""" + "".join(cards) + """<h2>Próxima decisão</h2><p>\nRevisar os cenários e os indícios dos erros; só depois definir o treino de controle.\nOs pesos antigos, o candidato e os testes anteriores permanecem preservados.</p></main></html>"""\n    (output_dir / "relatorio_diagnostico.html").write_text(html_report, encoding="utf-8")\n    lines = ["# Diagnóstico 11", "", "Comparação sem novo treinamento e sem seleção automática.",\n             "", f"Proveniência original do baseline: {provenance}.",\n             f"Validação externa: {len(new_records)} imagens.",\n             f"Reserva futura não inferida: {dataset_manifest[\'contagens\'].get(\'reserva_futura\', 0)} imagens.",\n             "", "| Conjunto | Modelo | Precisão alertas | Recall alertas | F1 | FN | Alta sem alerta |",\n             "|---|---|---:|---:|---:|---:|---:|"]\n    for r in comparison:\n        lines.append("| "+" | ".join(map(str, [r["coorte"], r["modelo"],\n                     format_percent(r["precision_alertas"]), format_percent(r["recall_alertas"]),\n                     format_percent(r["f1_alertas"]), r["fn_alerta"], r["alta_sem_alerta"]]))+" |")\n    lines += ["", "A nova cobertura é conservadora e a seleção busca diversidade de classes; não representa a frequência natural do uso.",\n              "A Home VAL continua exploratória enquanto a origem do primeiro treinamento não estiver esclarecida.",\n              "Semelhança por pHash é um filtro de suspeitas, não uma prova de identidade de cena.",\n              "Os indícios de erro podem se sobrepor e não identificam causas de forma conclusiva.",\n              "A reserva atual não foi inferida; não há avaliação independente de risco físico.",\n              "", "Veja relatorio_diagnostico.html para a galeria e bootstrap_pareado.csv para a incerteza das diferenças."]\n    (output_dir / "relatorio_diagnostico.md").write_text("\\n".join(lines)+"\\n", encoding="utf-8")\n    return comparison, diagnosis, gallery\n\n\ndef package_reports(output_dir):\n    output_dir = Path(output_dir)\n    archive = output_dir / "pacote_diagnostico_11.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as z:\n        for path in sorted(output_dir.rglob("*")):\n            if path.is_file() and path != archive and not path.name.endswith(".tmp"):\n                z.write(path, path.relative_to(output_dir))\n    return archive\n', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'exp12_data.py': '"""Entradas congeladas e replay somente do TRAIN histórico."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nimport yaml\n\nimport diag11_core as core\nimport diag11_data as d11\nfrom exp12_protocol import diverse_subset\n\n\ndef check_digest(raw, expected, label):\n    if hashlib.sha256(raw).hexdigest() != expected:\n        raise RuntimeError("Conteúdo alterado: " + label)\n\n\ndef materialise_home_labels(zip_path, home):\n    with zipfile.ZipFile(zip_path) as z:\n        for record in home:\n            member = record["id"].removeprefix("home:")\n            label_name = member.replace("/images/", "/labels/").rsplit(".", 1)[0]+".txt"\n            core.safe_member(label_name)\n            raw = z.read(label_name)\n            check_digest(raw, record["label_sha256"], label_name)\n            label = Path(str(record["image"]).replace("/images/", "/labels/")).with_suffix(".txt")\n            label.parent.mkdir(parents=True, exist_ok=True)\n            label.write_bytes(raw)\n            record["label"] = label\n\n\ndef read_package11(path, expected_sha, expected_models, expected_dataset):\n    if core.sha256_file(path) != expected_sha:\n        raise RuntimeError("O pacote do diagnóstico 11 mudou.")\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Pacote 11 com membros repetidos.")\n        for name in z.namelist():\n            core.safe_member(name)\n        summary = json.loads(z.read("resumo_diagnostico_11.json"))\n        config = json.loads(z.read("configuracao_diagnostico.json"))\n        dataset_location = json.loads(z.read("localizacao_dataset.json"))\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if (config["modelos_sha256"] != expected_models or\n                config["dataset_sha256"] != expected_dataset or\n                dataset_location["dataset_sha256"] != expected_dataset or\n                manifest["dataset_sha256"] != expected_dataset or\n                summary["reserva_futura_inferida"] is not False):\n            raise RuntimeError("Identidades ou reserva do diagnóstico 11 incompatíveis.")\n    return summary, config, dataset_location, manifest\n\n\ndef load_frozen_validation(zip_path, destination, expected_archive_sha, expected_dataset_sha,\n                           expected_manifest):\n    if core.sha256_file(zip_path) != expected_archive_sha:\n        raise RuntimeError("ZIP da validação independente diferente do congelado no 11.")\n    destination = Path(destination)\n    val, reserved_ids, frozen_files = [], set(), []\n    with zipfile.ZipFile(zip_path) as z:\n        if len(z.namelist()) != len(set(z.namelist())):\n            raise RuntimeError("Dataset novo com membros repetidos.")\n        for member in z.namelist():\n            core.safe_member(member)\n        manifest = json.loads(z.read("manifesto_validacao_nova.json"))\n        if manifest != expected_manifest:\n            raise RuntimeError("O manifesto do pacote difere do diagnóstico 11.")\n        raw_manifest = dict(manifest)\n        claimed = raw_manifest.pop("dataset_sha256")\n        if hashlib.sha256(core.canonical(raw_manifest).encode()).hexdigest() != claimed or claimed != expected_dataset_sha:\n            raise RuntimeError("Hash canônico da validação nova não confere.")\n        if manifest["nomes"] != core.NAMES or manifest["origem_split"] != "validation":\n            raise RuntimeError("Classes ou origem da validação alteradas.")\n        items = manifest["arquivos"]\n        if (Counter(r["role"] for r in items) != {"val_nova": 1000, "reserva_futura": 300}\n                or len({r["id"] for r in items}) != 1300\n                or len({r["sha256"] for r in items}) != 1300):\n            raise RuntimeError("Identidades ou quantidades do conjunto independente incompatíveis.")\n        for item in items:\n            role = item["role"]\n            for field, prefix in [("image", "images"), ("label", "labels"), ("coverage", "coverage")]:\n                if not item[field].startswith(prefix+"/"+role+"/"):\n                    raise RuntimeError("Caminho do dataset não corresponde ao papel declarado.")\n                if item[field] not in z.namelist():\n                    raise RuntimeError("Arquivo declarado ausente no ZIP.")\n            if role == "reserva_futura":\n                reserved_ids.add(item["id"])\n                continue  # Nenhum JPEG, label ou cobertura da reserva é extraído.\n            paths = {}\n            for field, hash_field in [("image", "sha256"), ("label", "label_sha256"),\n                                      ("coverage", "coverage_sha256")]:\n                raw = z.read(item[field])\n                check_digest(raw, item[hash_field], item[field])\n                output = destination/item[field]\n                output.parent.mkdir(parents=True, exist_ok=True)\n                output.write_bytes(raw)\n                paths[field] = output\n                frozen_files.append((output, item[hash_field]))\n            coverage = json.loads(paths["coverage"].read_text())\n            if any(coverage[k] != item[k] for k in ["known_mask", "positive_mask", "negative_mask"]):\n                raise RuntimeError("Cobertura da validação foi alterada.")\n            with Image.open(paths["image"]) as image:\n                if image.size != (item["width"], item["height"]):\n                    raise RuntimeError("Dimensões diferentes do manifesto.")\n                image.verify()\n            parsed = core.parse_yolo(paths["label"].read_bytes(), item["width"], item["height"])\n            if len(parsed) != len(item["gt_normalizado"]):\n                raise RuntimeError("Quantidade de caixas difere do manifesto.")\n            gt = []\n            for original, label_obj in zip(item["gt_normalizado"], parsed):\n                a, b, c, d = original["bbox_norm"]\n                box = [a*item["width"], b*item["height"], c*item["width"], d*item["height"]]\n                if (original["classe_id"] != label_obj["classe_id"] or\n                        not np.allclose(box, label_obj["bbox"], rtol=0, atol=.001)):\n                    raise RuntimeError("Caixa do manifesto diverge do label autenticado.")\n                gt.append({**original, "bbox": box})\n            val.append({**item, **paths, "gt": gt, "role": "val_nova",\n                        "coorte": "openimages_val_nova", "name": paths["image"].name})\n    return val, reserved_ids, frozen_files\n\n\ndef load_replay(package_items, selected_history, local_dir, fetch_package, expected_cfg,\n                forbidden_ids, forbidden_hashes, protected_near, validator, final_size=2000, seed=42):\n    """Lê só os membros escolhidos dos cinco lotes existentes, com SHA integral."""\n    local_dir = Path(local_dir)\n    local_dir.mkdir(parents=True, exist_ok=True)\n    wanted = {r["id"]: r for r in selected_history}\n    protected_ids, protected_hashes = set(forbidden_ids), set(forbidden_hashes)\n    records, exclusions = {}, []\n    frozen, seen, package_receipts = [], set(), []\n    for item in package_items:\n        pkg = local_dir/item["arquivo"]\n        fetch_package(item, pkg)\n        if pkg.stat().st_size != item["zip_bytes"] or core.sha256_file(pkg) != item["zip_sha256"]:\n            raise RuntimeError("Pacote de replay incompleto ou alterado.")\n        with zipfile.ZipFile(pkg) as z:\n            if len(z.namelist()) != len(set(z.namelist())):\n                raise RuntimeError("Lote de replay com membros repetidos.")\n            for name in z.namelist():\n                core.safe_member(name)\n            manifest = json.loads(z.read("manifesto_lote.json"))\n            if manifest["config_sha256"] != expected_cfg or manifest["origem_split"] != "train":\n                raise RuntimeError("Lote de replay não pertence ao TRAIN autenticado.")\n            for r in manifest["aceitas"]:\n                if r["id"] not in wanted:\n                    continue\n                if r["id"] in seen:\n                    raise RuntimeError("ID de replay repetido entre lotes.")\n                seen.add(r["id"])\n                prior = wanted[r["id"]]\n                if (prior["split"] != "train" or prior["lote"] != item["arquivo"] or\n                        prior["imagem_sha256"] != r["sha256"] or\n                        int(prior["known_mask"]) != r["known_mask"]):\n                    raise RuntimeError("Replay diverge da divisão 80/20 histórica.")\n                if r["id"] in protected_ids or r["sha256"] in protected_hashes:\n                    raise RuntimeError("Replay coincide com dado protegido.")\n                raw_image, raw_label, raw_cov = (z.read(r[k]) for k in ["imagem", "label", "cobertura"])\n                for raw, key in [(raw_image, "sha256"), (raw_label, "label_sha256"), (raw_cov, "cobertura_sha256")]:\n                    check_digest(raw, r[key], r["id"]+"/"+key)\n                classes = validator(raw_label, source=item["arquivo"]+":"+r["label"])\n                cov = json.loads(raw_cov)\n                known_classes = (set(cov["classes_positivas_verificadas_exaustivas"]) |\n                                 set(cov["classes_negativas_verificadas"]))\n                if (not all(isinstance(cid, int) and 0 <= cid < 12 for cid in known_classes)\n                        or sum(1 << cid for cid in known_classes) != r["known_mask"]\n                        or cov["known_mask"] != r["known_mask"]\n                        or set(classes) & set(cov["classes_negativas_verificadas"])):\n                    raise RuntimeError("Máscara ou anotação parcial de replay incompatível.")\n                image = local_dir/"images/train"/f"oi_{r[\'id\']}.jpg"\n                label = local_dir/"labels/train"/f"oi_{r[\'id\']}.txt"\n                coverage_file = local_dir/"coverage/train"/f"oi_{r[\'id\']}.json"\n                for path, raw in [(image, raw_image), (label, raw_label), (coverage_file, raw_cov)]:\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(raw)\n                with Image.open(image) as im:\n                    width, height = im.size\n                    im.verify()\n                if (width, height) != (r["width"], r["height"]):\n                    raise RuntimeError("Dimensões de replay incompatíveis.")\n                near = protected_near.find(d11.phash(image))\n                if near:\n                    exclusions.append({"id": r["id"], "motivo": "semelhanca_phash_com_validacao_ou_reserva", "semelhante_a": near})\n                    continue\n                gt = core.parse_yolo(raw_label, width, height)\n                records[r["id"]] = {\n                    **prior, "oi_id": r["id"], "id": r["id"], "source": "openimages",\n                    "image": image, "label": label, "coverage": coverage_file,\n                    "known_mask": r["known_mask"], "sha256": r["sha256"],\n                    "label_sha256": r["label_sha256"], "coverage_sha256": r["cobertura_sha256"],\n                    "width": width, "height": height, "classes_com_caixa": classes,\n                    "gt": gt, "name": image.name, "role": "train", "coorte": "replay_train",\n                }\n        package_receipts.append(item)\n        pkg.unlink()\n    if seen != set(wanted):\n        raise RuntimeError("Alguns IDs planejados não foram encontrados nos cinco lotes.")\n    if len(records) < final_size:\n        raise RuntimeError(f"Replay com somente {len(records)} elegíveis; exigidos {final_size}.")\n    chosen = diverse_subset(list(records.values()), final_size, seed)\n    output = {Path(r["image"]).resolve().as_posix(): r for r in chosen}\n    for record in output.values():\n        frozen += [(record["image"], record["sha256"]), (record["label"], record["label_sha256"]),\n                   (record["coverage"], record["coverage_sha256"])]\n    return output, exclusions, frozen, package_receipts\n\n\ndef make_training_inputs(home, replay, destination):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    home_train = {Path(r["image"]).resolve().as_posix():\n                  {**r, "source": "home", "known_mask": core.ALL_KNOWN}\n                  for r in home if r["role"] == "home_train"}\n    val = [Path(r["image"]).resolve().as_posix() for r in home if r["role"] == "home_val"]\n    if len(home_train) != 1824 or len(val) != 456:\n        raise RuntimeError("Home diferente da configuração congelada.")\n    val_file = destination/"home_val.txt"\n    val_file.write_text("\\n".join(sorted(val))+"\\n", encoding="utf-8")\n    arms = {}\n    for name, records in [("A_home", home_train), ("B_home_replay", {**home_train, **replay})]:\n        path = destination/(name+".txt")\n        path.write_text("\\n".join(sorted(records))+"\\n", encoding="utf-8")\n        data_file = destination/(name+".yaml")\n        data_file.write_text(yaml.safe_dump({"path": str(destination.resolve()), "train": str(path.resolve()),\n                                           "val": str(val_file.resolve()), "names": dict(enumerate(core.NAMES))},\n                                          sort_keys=False, allow_unicode=True), encoding="utf-8")\n        arms[name] = {"records": records, "yaml": data_file}\n    return arms\n\n\ndef restore_result(package_path, cohort, model, cache_dir):\n    cache_dir = Path(cache_dir)/cohort/model\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(package_path) as z:\n        prefix = cohort+"/"+model+"/"\n        summary = json.loads(z.read(prefix+"metricas.json"))\n        def rows(name):\n            return list(csv.DictReader(io.StringIO(z.read(prefix+name).decode("utf-8-sig"))))\n        images, events, classes = rows("alertas_por_imagem.csv"), rows("avaliacao_objetos.csv"), rows("metricas_por_classe.csv")\n        for row in images:\n            for k in ["tp_alerta", "fn_alerta", "fp_alerta", "alta_ref", "alta_sem_alerta", "tp_detector",\n                      "fn_detector", "fp_detector", "gt_avaliaveis", "gt_ignorados", "predicoes_ignoradas_conf_025"]:\n                row[k] = int(row[k])\n            for k in ["cobertura_completa_alertas", "ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        for row in events:\n            row["classe_id"] = int(row["classe_id"])\n            for k in ["ha_alerta_ref", "ha_alerta_pred"]:\n                row[k] = row[k] == "True"\n        cache = cache_dir/"predicoes_conf_0001.jsonl.gz"\n        cache.write_bytes(z.read(prefix+"predicoes_conf_0001.jsonl.gz"))\n    return {"summary": summary, "images": images, "events": events, "classes": classes, "cache_file": cache}\n\n\ndef verify_frozen(files):\n    for path, sha in files:\n        if core.sha256_file(path) != sha:\n            raise RuntimeError("Arquivo protegido alterado: "+str(path))\n', 'audit13.py': '"""Auditoria 13: dados imutáveis, casos de treino/validação e revisão portátil."""\nfrom __future__ import annotations\nimport base64\nfrom collections import Counter,defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image,ImageDraw\nimport numpy as np\nimport diag11_core as core\nfrom diag11_reports import font\nfrom exp12_protocol import order_id\n\nTRAIN_ROLES={\'home_train\',\'train\'}\nVAL_ROLES={\'home_val\',\'val_nova\'}\nVERSION=\'13-auditoria-1.0\'\nREVIEW_VERSION=\'13-revisoes-1.0\'\nFIELDS={\n \'rotulo\':(\'Rótulo da caixa destacada\',[(\'coerente\',\'Coerente\'),(\'classe_inadequada\',\'Classe inadequada\'),(\'duvidoso\',\'Duvidoso\')]),\n \'caixa\':(\'Caixa de referência\',[(\'adequada\',\'Adequada\'),(\'parcial\',\'Parcial / imprecisa\'),(\'inadequada\',\'Inadequada\'),(\'duvidoso\',\'Duvidosa\')]),\n \'ambiente\':(\'Ambiente\',[(\'interno\',\'Interno\'),(\'externo\',\'Externo\'),(\'ambiguo\',\'Ambíguo\')]),\n \'tipo_imagem\':(\'Tipo de imagem\',[(\'foto\',\'Fotografia\'),(\'render_ilustracao\',\'Render ou ilustração\'),(\'duvidoso\',\'Duvidoso\')]),\n \'conceito\':(\'O que há na caixa destacada?\',[(x,x.replace(\'_\',\' \').capitalize()) for x in [\'guarda_roupa\',\'closet\',\'aparador\',\'rack\',\'estante\',\'armario_generico\',\'arara_de_roupas\',\'mesa\',\'cadeira\',\'outro\',\'varios_conceitos\',\'duvidoso\']]),\n \'causa\':(\'Diagnóstico visual do erro\',[(\'rotulo_ou_gt_duvidoso\',\'Rótulo ou caixa de referência duvidosos\'),(\'caixa_previsao\',\'Caixa da previsão desalinhada\'),(\'pequeno_ou_ocluido\',\'Objeto pequeno ou ocluído\'),(\'indicio_confianca_csv\',\'CSV indica detecção abaixo de 0,25\'),(\'omissao_sem_causa_confirmada\',\'Omissão sem causa confirmada\'),(\'sem_erro_no_foco\',\'Sem erro no objeto em foco\'),(\'duvidoso\',\'Duvidoso\'),(\'nao_aplica_treino\',\'Não se aplica: caso de treino\')]),\n \'destino_treino\':(\'Triagem para um recorte futuro\',[(\'manter_candidato\',\'Candidato a revisão completa\'),(\'revisar_anotacao\',\'Revisar anotação antes de considerar\'),(\'excluir_candidato\',\'Excluir do recorte proposto\'),(\'nao_aplica_validacao\',\'Não se aplica: validação\')]),\n}\n\ndef json_rows(z,name):\n    return list(csv.DictReader(io.StringIO(z.read(name).decode(\'utf-8-sig\'))))\n\ndef bool_value(x):\n    if isinstance(x,bool):return x\n    if x in {\'True\',\'False\'}:return x==\'True\'\n    raise ValueError(\'Booleano inválido nos eventos.\')\n\ndef parse_events(rows):\n    out=[]\n    for r in rows:\n        v=dict(r);v[\'classe_id\']=int(r[\'classe_id\'])\n        if not 0<=v[\'classe_id\']<12:raise ValueError(\'Classe inválida nos eventos.\')\n        for k in [\'bbox_ref\',\'bbox_pred\']:\n            v[k]=json.loads(r[k]) if r[k] else None\n            if v[k] is not None and (len(v[k])!=4 or not np.isfinite(v[k]).all()):raise ValueError(\'Caixa inválida.\')\n        for k in [\'ha_alerta_ref\',\'ha_alerta_pred\']:v[k]=bool_value(r[k])\n        v[\'confianca\']=float(r[\'confianca\']) if r[\'confianca\'] else None\n        out.append(v)\n    return out\n\ndef load_source12(path,expected_sha,expected_code):\n    if core.sha256_file(path)!=expected_sha:raise RuntimeError(\'O pacote 12 não é o recebido e analisado.\')\n    result={\'events\':{},\'images\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):raise RuntimeError(\'Membros repetidos no pacote 12.\')\n        for n in z.namelist():core.safe_member(n)\n        if z.testzip() is not None:raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        result[\'summary\']=json.loads(z.read(\'resumo_experimento_12.json\'))\n        result[\'config\']=json.loads(z.read(\'configuracao_experimento_12.json\'))\n        result[\'replay\']=json.loads(z.read(\'manifesto_replay.json\'))\n        if not result[\'summary\'][\'concluido\'] or result[\'summary\'][\'reserva_futura_inferida\']:\n            raise RuntimeError(\'Execução 12 incompatível com esta auditoria.\')\n        if result[\'config\'][\'codigo_embutido_sha256\']!=expected_code:raise RuntimeError(\'Código do 12 diferente do preparado.\')\n        for n,sha in expected_code.items():\n            if hashlib.sha256(z.read(\'codigo/\'+n)).hexdigest()!=sha:raise RuntimeError(\'Código alterado no pacote 12.\')\n        for cohort in [\'home_val_historica\',\'openimages_val_nova\']:\n            result[\'events\'][cohort]={};result[\'images\'][cohort]={}\n            for model,prefix in [(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n                base=prefix+\'/\'+cohort+\'/\'\n                metrics=json.loads(z.read(base+\'metricas.json\'))\n                images=json_rows(z,base+\'alertas_por_imagem.csv\')\n                for r in images:\n                    for k in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\']:r[k]=int(r[k])\n                if len(images)!=len({r[\'id\'] for r in images}) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou contagens inconsistentes nos registros do 12.\')\n                for k,v in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[k] for r in images)!=v:raise RuntimeError(\'Métricas divergem das contagens por imagem.\')\n                result[\'events\'][cohort][model]=parse_events(json_rows(z,base+\'avaliacao_objetos.csv\'))\n                result[\'images\'][cohort][model]={r[\'id\']:r for r in images}\n            if result[\'images\'][cohort][\'candidato_10\'].keys()!=result[\'images\'][cohort][\'B_e03\'].keys():\n                raise RuntimeError(\'As duas versões não avaliaram os mesmos IDs.\')\n    return result\n\ndef event_key(event):\n    return (event[\'id\'],event[\'classe_id\'],tuple(round(x,6) for x in event[\'bbox_ref\']))\n\ndef error_plan(bundle,seed=42):\n    plan=[];receipts=[]\n    for cohort,quotas in [\n        (\'home_val_historica\',{\'nova_omissao_alta\':4,\'perda_de_alerta\':2,\'omissao_persistente\':3,\'melhora\':2,\'controle\':1}),\n        (\'openimages_val_nova\',{\'nova_omissao_alta\':14,\'perda_de_alerta\':3,\'omissao_persistente\':3,\'melhora\':2,\'controle\':2})]:\n        old,new=[bundle[\'events\'][cohort][m] for m in [\'candidato_10\',\'B_e03\']]\n        old_gt=defaultdict(list);new_gt=defaultdict(list)\n        for r in old:\n            if r[\'bbox_ref\'] is not None:old_gt[event_key(r)].append(r)\n        for r in new:\n            if r[\'bbox_ref\'] is not None:new_gt[event_key(r)].append(r)\n        if Counter({k:len(v) for k,v in old_gt.items()})!=Counter({k:len(v) for k,v in new_gt.items()}):\n            raise RuntimeError(\'Referências de objetos diferentes entre as avaliações.\')\n        pools=defaultdict(list);duplicates=0\n        for key,old_rows in old_gt.items():\n            if len(old_rows)!=1:duplicates+=1;continue\n            a,b=old_rows[0],new_gt[key][0]\n            if a[\'classe_id\'] not in {2,3} or not a[\'ha_alerta_ref\']:continue\n            if a[\'ha_alerta_pred\'] and not b[\'ha_alerta_pred\']:\n                category=\'nova_omissao_alta\' if b[\'prioridade_ref\']==\'alto\' else \'perda_de_alerta\'\n            elif not a[\'ha_alerta_pred\'] and not b[\'ha_alerta_pred\']:category=\'omissao_persistente\'\n            elif not a[\'ha_alerta_pred\'] and b[\'ha_alerta_pred\']:category=\'melhora\'\n            else:category=\'controle\'\n            pools[category].append({\'coorte\':cohort,\'old\':a,\'new\':b,\'categoria\':category,\'key\':key})\n        for group in pools.values():group.sort(key=lambda r:order_id(core.canonical(r[\'key\']),seed))\n        chosen=[];seen=set();seen_images=set()\n        def take(group,n,unique_images=True):\n            added=0\n            for item in pools[group]:\n                if added>=n:break\n                if item[\'key\'] in seen or (unique_images and item[\'old\'][\'id\'] in seen_images):continue\n                chosen.append(item);seen.add(item[\'key\']);seen_images.add(item[\'old\'][\'id\']);added+=1\n            return added\n        for group,n in quotas.items():take(group,n)\n        for distinct in [True,False]:\n            for group in quotas:take(group,sum(quotas.values())-len(chosen),distinct)\n        plan+=chosen\n        receipts.append({\'coorte\':cohort,\'limite\':sum(quotas.values()),\'selecionados\':len(chosen),\n            \'disponiveis_por_categoria\':{k:len(v) for k,v in pools.items()},\n            \'selecionados_por_categoria\':dict(Counter(r[\'categoria\'] for r in chosen)),\n            \'referencias_duplicadas_ambiguas_excluidas\':duplicates})\n    return plan,receipts\n\ndef replay_plan(manifest,history,seed=42):\n    rows=manifest[\'imagens\'];byid={r[\'id\']:r for r in history[\'rows\']}\n    if len(rows)!=2000 or len({r[\'id\'] for r in rows})!=2000:raise RuntimeError(\'Replay 12 diferente do congelado.\')\n    for r in rows:\n        prior=byid[r[\'id\']]\n        if prior[\'split\']!=\'train\' or prior[\'imagem_sha256\']!=r[\'sha256\'] or prior[\'lote\']!=r[\'lote\']:\n            raise RuntimeError(\'O replay não corresponde ao TRAIN histórico.\')\n    selected=[]\n    for cid,n in [(7,24),(3,6),(2,6)]:\n        group=sorted([r for r in rows if int(r[\'classe_primaria\'])==cid],key=lambda r:order_id(r[\'id\'],seed))\n        if len(group)<n:raise RuntimeError(\'Suporte insuficiente na amostra de treino para auditoria.\')\n        selected += [byid[r[\'id\']] for r in group[:n]]\n    return selected\n\ndef focus_index(record,cid):\n    candidates=[i for i,o in enumerate(record[\'gt\']) if o[\'classe_id\']==cid]\n    if not candidates:raise RuntimeError(\'Classe selecionada sem caixa no label original.\')\n    return min(candidates,key=lambda i:(-(record[\'gt\'][i][\'bbox\'][2]-record[\'gt\'][i][\'bbox\'][0])*(record[\'gt\'][i][\'bbox\'][3]-record[\'gt\'][i][\'bbox\'][1]),i))\n\ndef make_case(record,index,group,reason,context=None):\n    if record[\'role\'] not in TRAIN_ROLES|VAL_ROLES:raise RuntimeError(\'Reserva ou teste entrou nos casos.\')\n    obj=record[\'gt\'][index]\n    identity=[record[\'id\'],record[\'role\'],record[\'sha256\'],obj[\'classe_id\'],obj[\'bbox\']]\n    cid=hashlib.sha256(core.canonical(identity).encode()).hexdigest()[:24]\n    return {\'case_id\':cid,\'id\':record[\'id\'],\'imagem\':record[\'name\'],\'papel\':record[\'role\'],\n            \'fonte\':\'Home\' if record[\'role\'].startswith(\'home_\') else \'Open Images\',\n            \'coorte\':record[\'role\'] if record[\'role\'] in TRAIN_ROLES else record[\'coorte\'],\'grupo\':group,\'motivo\':reason,\'classe_id\':obj[\'classe_id\'],\n            \'classe\':core.NAMES[obj[\'classe_id\']],\'gt_index\':index,\'bbox_ref\':obj[\'bbox\'],\n            \'width\':record[\'width\'],\'height\':record[\'height\'],\'known_mask\':record[\'known_mask\'],\n            \'image_sha256\':record[\'sha256\'],\'label_sha256\':record[\'label_sha256\'],\n            \'arquivo\':\'galeria/\'+cid+\'.jpg\',\'contexto\':context or {}}\n\ndef build_cases(home,replay,val,plan,seed=42):\n    cases=[];records={r[\'id\']:r for r in home+list(replay.values())+val}\n    for source,items in [(\'Home\',[r for r in home if r[\'role\']==\'home_train\']),(\'Open Images\',list(replay.values()))]:\n        for cid,n in [(7,24),(3,6),(2,6)]:\n            candidates=[r for r in items if any(o[\'classe_id\']==cid for o in r[\'gt\'])]\n            if source==\'Open Images\':candidates=[r for r in candidates if int(r[\'classe_primaria\'])==cid]\n            candidates.sort(key=lambda r:order_id(r[\'id\'],seed))\n            if len(candidates)<n:raise RuntimeError(\'Dados de treino insuficientes para a auditoria planejada.\')\n            for r in candidates[:n]:\n                cases.append(make_case(r,focus_index(r,cid),\'semantica_treino\' if cid==7 else \'rotulos_treino\',\n                    \'Amostra por hash de imagens com a classe; maior caixa da classe em foco.\'))\n    for item in plan:\n        r=records[item[\'old\'][\'id\']]\n        expected=\'home_val\' if item[\'coorte\']==\'home_val_historica\' else \'val_nova\'\n        if r[\'role\']!=expected:raise RuntimeError(\'Papel inválido para revisão de erros.\')\n        indices=[i for i,g in enumerate(r[\'gt\']) if g[\'classe_id\']==item[\'old\'][\'classe_id\'] and np.allclose(g[\'bbox\'],item[\'old\'][\'bbox_ref\'],rtol=0,atol=.001)]\n        if len(indices)!=1:raise RuntimeError(\'Objeto do registro não corresponde univocamente ao GT autenticado.\')\n        cases.append(make_case(r,indices[0],\'erros_validacao\',item[\'categoria\'],{\n            \'evento_candidato10\':item[\'old\'][\'evento\'],\'evento_B_e03\':item[\'new\'][\'evento\'],\n            \'alerta_candidato10\':item[\'old\'][\'ha_alerta_pred\'],\'alerta_B_e03\':item[\'new\'][\'ha_alerta_pred\'],\n            \'prioridade_ref\':item[\'old\'][\'prioridade_ref\'],\'indicios_candidato10\':item[\'old\'][\'indicios\'],\n            \'indicios_B_e03\':item[\'new\'][\'indicios\']}))\n    if len({c[\'case_id\'] for c in cases})!=len(cases):raise RuntimeError(\'Casos duplicados na amostra.\')\n    group_order={\'semantica_treino\':0,\'rotulos_treino\':1,\'erros_validacao\':2}\n    return sorted(cases,key=lambda c:(group_order[c[\'grupo\']],c[\'fonte\'],c[\'classe_id\'],c[\'case_id\'])),records\n\ndef predictions_from_events(events):\n    result=defaultdict(list);seen=set()\n    for e in events:\n        if e[\'bbox_pred\'] is None:continue\n        if e[\'confianca\'] is None or e[\'confianca\']<.25:raise RuntimeError(\'Previsão sem confiança mínima nos registros.\')\n        key=(e[\'id\'],e[\'classe_id\'],tuple(e[\'bbox_pred\']),e[\'confianca\'])\n        if key in seen:continue\n        seen.add(key);result[e[\'id\']].append({\'classe_id\':e[\'classe_id\'],\'classe\':e[\'classe\'],\n            \'bbox\':e[\'bbox_pred\'],\'confianca\':e[\'confianca\']})\n    return result\n\ndef draw_panel(record,objects,title,focus=None,predictions=False):\n    canvas=Image.new(\'RGB\',(540,480),\'#111c2c\');d=ImageDraw.Draw(canvas)\n    d.text((12,10),title,fill=\'white\',font=font(15))\n    with Image.open(record[\'image\']) as image:\n        image=image.convert(\'RGB\');scale=min(520/image.width,390/image.height)\n        image=image.resize((max(1,round(image.width*scale)),max(1,round(image.height*scale))),Image.Resampling.LANCZOS)\n    x0,y0=(540-image.width)//2,48+(390-image.height)//2;canvas.paste(image,(x0,y0))\n    sx,sy=image.width/record[\'width\'],image.height/record[\'height\']\n    ordered=[i for i in range(len(objects)) if i!=focus]+([focus] if focus is not None else [])\n    for i in ordered:\n        o=objects[i];box=[x0+o[\'bbox\'][0]*sx,y0+o[\'bbox\'][1]*sy,x0+o[\'bbox\'][2]*sx,y0+o[\'bbox\'][3]*sy]\n        color=\'#c09bff\' if predictions else (\'#58e8df\' if i==focus else \'#9ca9bb\')\n        d.rectangle(box,outline=color,width=4 if i==focus else 2)\n        label=o[\'classe\']+(f" {o[\'confianca\']:.2f}" if predictions else (\' / FOCO\' if i==focus else \'\'))\n        tx=max(1,min(box[0],460));ty=max(43,box[1]-16)\n        d.rectangle(d.textbbox((tx,ty),label,font=font(11)),fill=\'#111c2c\');d.text((tx,ty),label,fill=color,font=font(11))\n    d.text((12,454),\'Roxo: previsões avaliáveis @0,25\' if predictions else \'Ciano: caixa de referência a revisar | cinza: demais rótulos\',fill=\'#d3ddeb\',font=font(10))\n    return canvas\n\ndef render_cases(cases,records,bundle,output):\n    output=Path(output);pred={}\n    for cohort,models in bundle[\'events\'].items():\n        pred[cohort]={m:predictions_from_events(events) for m,events in models.items()}\n    for i,case in enumerate(cases,1):\n        record=records[case[\'id\']]\n        panels=[draw_panel(record,record[\'gt\'],\'Rótulos originais / \'+case[\'fonte\'],case[\'gt_index\'])]\n        if case[\'papel\'] in VAL_ROLES:\n            for model,title in [(\'candidato_10\',\'Candidato 10 / registros do experimento 12\'),(\'B_e03\',\'Home + replay / época 3 do experimento 12\')]:\n                panels.append(draw_panel(record,pred[case[\'coorte\']][model].get(case[\'id\'],[]),title,predictions=True))\n        image=Image.new(\'RGB\',(540*len(panels),480),\'#111c2c\')\n        for n,panel in enumerate(panels):image.paste(panel,(540*n,0))\n        path=output/case[\'arquivo\'];path.parent.mkdir(parents=True,exist_ok=True);image.save(path,quality=86)\n        if i%20==0 or i==len(cases):print(\'Casos renderizados:\',i,\'/\',len(cases),flush=True)\n\ndef make_manifest(cases,inputs,receipts,seed=42):\n    manifest={\'versao\':VERSION,\'inputs\':inputs,\'seed\':seed,\'casos\':cases,\'selecao_erros\':receipts,\n        \'amostra_dirigida_nao_estima_prevalencia\':True,\'validacao_nunca_vira_treino\':True,\n        \'inferencias_novas\':0,\'novos_treinamentos\':0,\'reserva_futura_extraida\':False,\'reserva_futura_inferida\':False,\n        \'limite_galeria\':\'Previsões em classes avaliáveis registradas nos CSVs a partir de confiança 0,25. Sem previsões de classes sem cobertura ou de confiança inferior.\'}\n    manifest[\'audit_manifest_sha256\']=hashlib.sha256(core.canonical(manifest).encode()).hexdigest()\n    return manifest\n\ndef empty_answer(case):\n    answer={\'case_id\':case[\'case_id\'],\'status\':\'pendente\',**{k:\'pendente\' for k in FIELDS},\'observacao\':\'\'}\n    answer[\'causa\']=\'nao_aplica_treino\' if case[\'papel\'] in TRAIN_ROLES else \'pendente\'\n    answer[\'destino_treino\']=\'nao_aplica_validacao\' if case[\'papel\'] in VAL_ROLES else \'pendente\'\n    return answer\n\ndef empty_review(manifest):\n    return {\'versao\':REVIEW_VERSION,\'audit_manifest_sha256\':manifest[\'audit_manifest_sha256\'],\n            \'responsavel\':\'\',\'wardrobe_decisao\':\'pendente\',\'observacao_taxonomia\':\'\',\n            \'casos\':[empty_answer(c) for c in manifest[\'casos\']]}\n\ndef validate_manifest(manifest):\n    raw=dict(manifest);claimed=raw.pop(\'audit_manifest_sha256\',None)\n    if hashlib.sha256(core.canonical(raw).encode()).hexdigest()!=claimed:\n        raise ValueError(\'Manifesto da auditoria alterado.\')\n    if len({c[\'case_id\'] for c in manifest[\'casos\']})!=len(manifest[\'casos\']):\n        raise ValueError(\'Manifesto com casos repetidos.\')\n    for c in manifest[\'casos\']:\n        if c[\'papel\'] not in TRAIN_ROLES|VAL_ROLES:raise ValueError(\'Reserva ou teste no manifesto.\')\n        identity=[c[\'id\'],c[\'papel\'],c[\'image_sha256\'],c[\'classe_id\'],c[\'bbox_ref\']]\n        if hashlib.sha256(core.canonical(identity).encode()).hexdigest()[:24]!=c[\'case_id\']:\n            raise ValueError(\'Identidade do caso alterada.\')\n\n\ndef validate_review(manifest,review):\n    validate_manifest(manifest)\n    if review.get(\'versao\')!=REVIEW_VERSION or review.get(\'audit_manifest_sha256\')!=manifest[\'audit_manifest_sha256\']:\n        raise ValueError(\'Revisão pertence a outra amostra ou versão de auditoria.\')\n    if review.get(\'wardrobe_decisao\') not in {\'pendente\',\'guarda_roupa_closet\',\'armazenamento_amplo\',\'separar_classes\'}:\n        raise ValueError(\'Decisão de taxonomia inválida.\')\n    for field in [\'responsavel\',\'observacao_taxonomia\']:\n        if not isinstance(review.get(field),str) or len(review[field])>10000:raise ValueError(\'Texto de revisão inválido.\')\n    cases={c[\'case_id\']:c for c in manifest[\'casos\']};answers={}\n    for row in review.get(\'casos\',[]):\n        cid=row.get(\'case_id\')\n        if cid not in cases or cid in answers:raise ValueError(\'Caso desconhecido ou repetido na revisão.\')\n        if set(row)!=set(empty_answer(cases[cid])):raise ValueError(\'Campos de revisão diferentes do formulário.\')\n        if row[\'status\'] not in {\'pendente\',\'revisado\'}:raise ValueError(\'Status de revisão inválido.\')\n        if not isinstance(row[\'observacao\'],str) or len(row[\'observacao\'])>10000:raise ValueError(\'Observação inválida.\')\n        for key,(_,options) in FIELDS.items():\n            if row[key] not in {\'pendente\'}|{x[0] for x in options}:raise ValueError(\'Resposta fora das opções: \'+key)\n        case=cases[cid]\n        if case[\'papel\'] in VAL_ROLES and row[\'destino_treino\']!=\'nao_aplica_validacao\':\n            raise ValueError(\'Validação não pode ser destinada a treino, mesmo por edição do JSON.\')\n        if case[\'papel\'] in TRAIN_ROLES:\n            if row[\'causa\']!=\'nao_aplica_treino\' or row[\'destino_treino\']==\'nao_aplica_validacao\':raise ValueError(\'Papel de treino inconsistente.\')\n        elif row[\'causa\']==\'nao_aplica_treino\':raise ValueError(\'Causa de treino em validação.\')\n        if row[\'causa\']==\'indicio_confianca_csv\' and not any(\'deteccao_abaixo_da_confianca_025\' in str(v) for v in case[\'contexto\'].values()):\n            raise ValueError(\'O CSV não registra esse indício de confiança para o caso.\')\n        if row[\'status\']==\'revisado\' and (any(row[k]==\'pendente\' for k in FIELDS) or not review[\'responsavel\'].strip()):\n            raise ValueError(\'Caso marcado revisado sem respostas completas ou responsável.\')\n        answers[cid]=row\n    if set(answers)!=set(cases):raise ValueError(\'A revisão precisa conter todos os casos, inclusive os pendentes.\')\n    return answers\n\ndef write_review_outputs(manifest,review,output):\n    output=Path(output);answers=validate_review(manifest,review);rows=[];triage=[]\n    for case in manifest[\'casos\']:\n        a=answers[case[\'case_id\']]\n        row={k:case[k] for k in [\'case_id\',\'id\',\'fonte\',\'papel\',\'classe\',\'image_sha256\',\'label_sha256\']}\n        row.update(a);rows.append(row)\n        if case[\'papel\'] in TRAIN_ROLES:\n            eligible=(a[\'status\']==\'revisado\' and a[\'rotulo\']==\'coerente\' and a[\'caixa\']==\'adequada\' and\n                a[\'ambiente\']==\'interno\' and a[\'tipo_imagem\']==\'foto\' and a[\'destino_treino\']==\'manter_candidato\')\n            if case[\'classe_id\']==7:\n                allowed={\'guarda_roupa\',\'closet\'} if review[\'wardrobe_decisao\']==\'guarda_roupa_closet\' else {\'guarda_roupa\',\'closet\',\'aparador\',\'rack\',\'estante\',\'armario_generico\',\'arara_de_roupas\'} if review[\'wardrobe_decisao\']==\'armazenamento_amplo\' else set()\n                eligible=eligible and a[\'conceito\'] in allowed\n            triage.append({**row,\'candidato_a_revisao_completa\':bool(eligible),\n                \'limite\':\'Somente a caixa em foco foi revisada; não é uma imagem inteira aprovada nem um dataset pronto.\'})\n    complete=sum(a[\'status\']==\'revisado\' for a in answers.values())\n    status={\'geracao_concluida\':True,\'casos\':len(rows),\'casos_revisados\':complete,\'casos_pendentes\':len(rows)-complete,\n        \'revisao_humana_concluida\':complete==len(rows) and review[\'wardrobe_decisao\']!=\'pendente\',\n        \'decisao_wardrobe\':review[\'wardrobe_decisao\'],\'objetos_treino_candidatos\':sum(r[\'candidato_a_revisao_completa\'] for r in triage),\n        \'novo_dataset_pronto\':False,\'rotulos_alterados\':False,\'novos_treinamentos\':0,\'inferencias_novas\':0,\n        \'reserva_futura_extraida\':False,\'reserva_futura_inferida\':False,\n        \'validacao_excluida_de_propostas_treino\':True,\'audit_manifest_sha256\':manifest[\'audit_manifest_sha256\']}\n    core.save_csv(output/\'revisoes_13.csv\',rows)\n    core.save_csv(output/\'triagem_objetos_treino.csv\',triage)\n    core.save_json(output/\'revisoes_13.json\',review)\n    core.save_json(output/\'resumo_auditoria_13.json\',status)\n    core.save_json(output/\'proposta_taxonomia.json\',{\'classe_atual\':\'wardrobe\',\'classe_id\':7,\n        \'decisao\':review[\'wardrobe_decisao\'],\'observacao\':review[\'observacao_taxonomia\'],\n        \'aplicada\':False,\'observacao_tecnica\':\'Ampliar ou separar a classe exige reanotação e nova versão do mapeamento/cobertura. Nenhuma anotação ou validação histórica é corrigida silenciosamente.\'})\n    return status\n\ndef build_reviewer(manifest,review,template,output):\n    output=Path(output);validate_review(manifest,review)\n    cards=[]\n    for case in manifest[\'casos\']:\n        raw=(output/case[\'arquivo\']).read_bytes()\n        cards.append({**case,\'image_data\':\'data:image/jpeg;base64,\'+base64.b64encode(raw).decode()})\n    payload={\'manifest_sha\':manifest[\'audit_manifest_sha256\'],\'review\':review,\'cases\':cards,\'fields\':FIELDS}\n    encoded=json.dumps(payload,ensure_ascii=False).replace(\'<\',\'\\\\u003c\').replace(\'&\',\'\\\\u0026\').replace(\'\\u2028\',\'\\\\u2028\').replace(\'\\u2029\',\'\\\\u2029\')\n    if template.count(\'__PAYLOAD__\')!=1:raise ValueError(\'Template do revisor inválido.\')\n    path=output/\'revisor_13.html\';path.write_text(template.replace(\'__PAYLOAD__\',encoded),encoding=\'utf-8\')\n    return path\n\ndef package_reports(output):\n    output=Path(output);path=output/\'pacote_auditoria_13.zip\'\n    with zipfile.ZipFile(path,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for p in sorted(output.rglob(\'*\')):\n            if p.is_file() and p!=path and not p.name.endswith(\'.tmp\'):\n                z.write(p,p.relative_to(output))\n    return path\n\n\ndef class_differences(bundle):\n    rows=[]\n    for cohort,models in bundle[\'events\'].items():\n        metrics={}\n        for model,events in models.items():\n            for cid in range(12):\n                es=[e for e in events if e[\'classe_id\']==cid]\n                metrics[model,cid]={\n                    \'tp\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in es),\n                    \'fn\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in es),\n                    \'fp\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in es),\n                    \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in es)}\n            images=bundle[\'images\'][cohort][model].values()\n            for k,col in [(\'tp\',\'tp_alerta\'),(\'fn\',\'fn_alerta\'),(\'fp\',\'fp_alerta\'),(\'alta_sem_alerta\',\'alta_sem_alerta\')]:\n                if sum(metrics[model,cid][k] for cid in range(12))!=sum(r[col] for r in images):\n                    raise RuntimeError(\'Eventos não reproduzem as métricas por imagem.\')\n        for cid in range(12):\n            row={\'coorte\':cohort,\'classe\':core.NAMES[cid],\'classe_id\':cid}\n            for k in [\'tp\',\'fn\',\'fp\',\'alta_sem_alerta\']:\n                a,b=metrics[\'candidato_10\',cid][k],metrics[\'B_e03\',cid][k]\n                row.update({k+\'_candidato10\':a,k+\'_B_e03\':b,\'delta_\'+k:b-a})\n            rows.append(row)\n    return rows\n', 'revisor_template.html': '<!doctype html>\n<html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>TCC — Revisor da auditoria 13</title>\n<style>\n:root{font-family:system-ui,sans-serif;color:#17263d;background:#f1f4f8;font-size:15px}*{box-sizing:border-box}body{margin:0}header{background:#12243e;color:white;padding:22px 5vw}header p{max-width:1050px;color:#d4deec;margin-bottom:0}h1{font-size:26px;line-height:1.2;margin:0}main{max-width:1380px;margin:auto;padding:20px}section,.toolbar{background:white;border:1px solid #d8e0eb;border-radius:10px;padding:18px;margin-bottom:16px}.toolbar{position:sticky;top:0;z-index:2;display:flex;flex-wrap:wrap;gap:10px;align-items:center;box-shadow:0 3px 12px #1231}button,.file-label{font:inherit;border:1px solid #b3c1d4;background:#f5f8fc;color:#173456;padding:10px 14px;border-radius:6px;cursor:pointer;display:inline-block}button.primary{background:#176b80;color:white;border-color:#176b80}button:disabled{opacity:.45;cursor:default}select,input,textarea{font:inherit;padding:9px;border:1px solid #adbed1;border-radius:5px;background:white;max-width:100%;width:100%}label{display:block;font-weight:600;font-size:13px;margin-bottom:5px}.grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}.filters{display:flex;gap:12px;flex-wrap:wrap;margin-bottom:15px}.filters>div{min-width:150px;flex:1}h2{font-size:20px;margin:0 0 8px}p{line-height:1.55}.muted{font-size:13px;color:#50637c}.notice{background:#eaf3f6;padding:12px;border-left:4px solid #176b80;border-radius:5px}.badge{font-size:12px;background:#e3ebf6;padding:4px 8px;border-radius:5px;display:inline-block;margin:0 4px 6px 0}.evidence{text-align:center;background:#111c2c;border-radius:6px;overflow:auto;margin:15px 0}.evidence img{max-width:100%;max-height:62vh;width:auto;cursor:zoom-in;vertical-align:middle}.navigation{display:flex;align-items:center;gap:10px;margin-top:15px;flex-wrap:wrap}.navigation select{width:auto;flex:1}textarea{min-height:75px;resize:vertical}#progress{font-weight:600;margin-right:auto}#message{min-height:20px;margin:8px 0;color:#8b3b15}details summary{cursor:pointer;font-weight:600}dialog{border:0;padding:10px;max-width:98vw;max-height:98vh;background:#111c2c}dialog::backdrop{background:#09131de0}dialog{width:96vw;height:92vh}dialog img{max-width:none;max-height:none;display:block}.zoomtools{position:sticky;top:0;left:0;background:#111c2c;color:white;display:flex;justify-content:space-between;gap:12px;align-items:center;margin-bottom:8px}.zoomtools button{margin:0}.smallcode{font:12px ui-monospace,monospace;overflow-wrap:anywhere}@media(max-width:800px){.grid{grid-template-columns:1fr}.toolbar{position:static}header{padding:20px}.evidence img{max-height:none}}\n</style>\n<header><h1>Auditoria 13 · rótulos e erros prioritários</h1><p>Uma caixa em foco por caso. Revise o significado do objeto, a anotação e o tipo de cena. Os dados originais permanecem preservados; suas respostas formam uma proposta documentada.</p></header>\n<main>\n<div class="toolbar"><span id="progress"></span><button id="export" class="primary">Baixar revisões em JSON</button><button id="exportcsv">Baixar CSV</button><label class="file-label" for="importfile" style="margin:0;font-weight:400">Importar revisões</label><input id="importfile" type="file" accept=".json,application/json" hidden></div>\n<section><div class="grid"><div><label for="reviewer">Responsável pela revisão</label><input id="reviewer" placeholder="Nome ou identificação do grupo"></div><div><label for="taxonomy">Definição proposta para wardrobe</label><select id="taxonomy"><option value="pendente">Pendente — decidir com o grupo</option><option value="guarda_roupa_closet">Guarda-roupa e closet</option><option value="armazenamento_amplo">Armazenamento amplo: armários, racks, estantes</option><option value="separar_classes">Separar conceitos em classes diferentes</option></select></div><div><label for="taxnote">Justificativa da definição</label><input id="taxnote" placeholder="Explique a decisão; nada será reanotado automaticamente"></div></div>\n<p class="notice">Os casos de validação servem para diagnosticar erros e nunca entram na proposta de treino. Nos casos de treino, a revisão cobre somente a caixa em foco: uma resposta favorável não aprova todos os rótulos da imagem.</p>\n<details><summary>Como usar e interpretar</summary><p>Preencha as opções do caso e clique em <strong>Marcar revisado e avançar</strong>. “Duvidoso” é uma resposta válida. A caixa em ciano é a referência em foco. Um objeto pequeno ou ocluído não tem, por isso só, um rótulo incorreto; avalie a classe e a caixa separadamente. Nos casos de validação, os painéis roxos mostram somente previsões em classes avaliáveis presentes nos CSVs do experimento 12, a partir de confiança 0,25. Previsões sem cobertura e abaixo desse limiar não estão desenhadas.</p><p>A amostra é dirigida para revisão de rótulos e diferenças entre checkpoints. Ela não estima a frequência dos problemas na base inteira. A política V2 não constitui uma avaliação independente de risco físico. Sua decisão de taxonomia ainda precisa virar uma versão documentada dos dados antes de qualquer novo treino.</p><p>As respostas são guardadas neste navegador quando possível. Use <strong>Baixar revisões em JSON</strong> para preservar e enviar o trabalho. Você pode exportar uma revisão parcial. Importar JSON permite continuar em outro computador ou numa nova execução da mesma amostra.</p></details>\n</section>\n<section>\n<div class="filters"><div><label for="groupfilter">Grupo</label><select id="groupfilter"><option value="todos">Todos</option><option value="semantica_treino">Guarda-roupa: treino</option><option value="rotulos_treino">Mesas e cadeiras: treino</option><option value="erros_validacao">Erros e controles: validação</option></select></div><div><label for="sourcefilter">Origem</label><select id="sourcefilter"><option value="todos">Todas</option><option value="Home">Home</option><option value="Open Images">Open Images</option></select></div><div><label for="statusfilter">Situação</label><select id="statusfilter"><option value="todos">Todos</option><option value="pendente">Pendentes</option><option value="revisado">Revisados</option></select></div></div>\n<div id="message" role="status" aria-live="polite"></div>\n<div id="casebody"><h2 id="caseheading"></h2><div id="badges"></div><p id="reason"></p><p id="context" class="muted"></p><div class="evidence"><img id="caseimage" alt="Referência original e, nos casos de validação, previsões registradas dos dois checkpoints" tabindex="0"></div><p class="muted">Clique na imagem para ampliar. Ciano: referência em foco. Cinza: demais rótulos. Roxo: previsões avaliáveis registradas.</p><div id="fields" class="grid"></div><div style="margin-top:15px"><label for="observation">Observação do caso</label><textarea id="observation" placeholder="O que precisa ser corrigido ou verificado? Não é necessário adivinhar a causa."></textarea></div><p id="caseid" class="smallcode muted"></p></div>\n<div class="navigation"><button id="prev">← Anterior</button><select id="jump" aria-label="Ir para caso"></select><button id="next">Próximo →</button><button id="complete" class="primary">Marcar revisado e avançar</button><button id="pending">Deixar pendente</button></div>\n</section><p class="muted">Nenhuma imagem ou resposta é enviada por esta página. O JSON exportado será validado pelo notebook antes de gerar qualquer triagem de treino.</p>\n</main><dialog id="zoom"><div class="zoomtools"><span>Resolução original · role para percorrer os painéis</span><button id="closezoom">Fechar ampliação</button></div><img id="zoomimage" alt="Imagem ampliada do caso"></dialog>\n<script id="payload" type="application/json">__PAYLOAD__</script>\n<script>\n\'use strict\';\nconst DATA=JSON.parse(document.getElementById(\'payload\').textContent);\nconst $=id=>document.getElementById(id);\nconst TRAIN=new Set([\'home_train\',\'train\']);\nconst allowedTax=new Set([\'pendente\',\'guarda_roupa_closet\',\'armazenamento_amplo\',\'separar_classes\']);\nlet review=structuredClone(DATA.review),visible=[],position=0,current=null;\nconst storageKey=\'tcc_auditoria13_\'+DATA.manifest_sha;\nfunction message(text){$(\'message\').textContent=text;}\nfunction answers(){return new Map(review.casos.map(r=>[r.case_id,r]));}\nfunction validate(candidate){\n if(candidate.versao!==\'13-revisoes-1.0\'||candidate.audit_manifest_sha256!==DATA.manifest_sha)throw Error(\'Este JSON pertence a outra amostra ou versão.\');\n if(!allowedTax.has(candidate.wardrobe_decisao)||typeof candidate.responsavel!==\'string\'||typeof candidate.observacao_taxonomia!==\'string\')throw Error(\'Metadados da revisão inválidos.\');\n if(!Array.isArray(candidate.casos)||candidate.casos.length!==DATA.cases.length)throw Error(\'A revisão deve incluir todos os casos, mesmo os pendentes.\');\n const byid=new Map(DATA.cases.map(c=>[c.case_id,c])),seen=new Set();\n for(const row of candidate.casos){\n  const c=byid.get(row.case_id);if(!c||seen.has(row.case_id))throw Error(\'Caso desconhecido ou repetido.\');seen.add(row.case_id);\n  const keys=[\'case_id\',\'status\',...Object.keys(DATA.fields),\'observacao\'];\n  if(Object.keys(row).length!==keys.length||keys.some(k=>!(k in row)))throw Error(\'Campos diferentes do formulário.\');\n  if(![\'pendente\',\'revisado\'].includes(row.status)||typeof row.observacao!==\'string\'||row.observacao.length>10000)throw Error(\'Resposta inválida.\');\n  for(const [k,entry] of Object.entries(DATA.fields))if(row[k]!==\'pendente\'&&!entry[1].some(o=>o[0]===row[k]))throw Error(\'Opção inválida: \'+k);\n  if(!TRAIN.has(c.papel)&&row.destino_treino!==\'nao_aplica_validacao\')throw Error(\'Validação não pode entrar no treino.\');\n  if(TRAIN.has(c.papel)&&(row.causa!==\'nao_aplica_treino\'||row.destino_treino===\'nao_aplica_validacao\'))throw Error(\'Resposta incompatível com o papel de treino.\');\n  if(!TRAIN.has(c.papel)&&row.causa===\'nao_aplica_treino\')throw Error(\'Diagnóstico incompatível com validação.\');\n  if(row.causa===\'indicio_confianca_csv\'&&!Object.values(c.contexto).some(v=>String(v).includes(\'deteccao_abaixo_da_confianca_025\')))throw Error(\'O CSV não registra detecção abaixo do limiar para este caso.\');\n  if(row.status===\'revisado\'&&(!candidate.responsavel.trim()||Object.keys(DATA.fields).some(k=>row[k]===\'pendente\')))throw Error(\'Preencha as opções e o responsável antes de marcar revisado.\');\n }\n return true;\n}\ntry{const saved=localStorage.getItem(storageKey);if(saved){const parsed=JSON.parse(saved);validate(parsed);review=parsed;}}catch(e){message(\'Não foi possível recuperar respostas locais. Você pode importar um JSON exportado.\');}\nfunction save(){try{localStorage.setItem(storageKey,JSON.stringify(review));}catch(e){message(\'Salvamento no navegador indisponível. Baixe o JSON para preservar as respostas.\');}progress();}\nfunction progress(){const done=review.casos.filter(r=>r.status===\'revisado\').length;$(\'progress\').textContent=done+\' de \'+DATA.cases.length+\' casos revisados\';}\nfunction metadata(){review.responsavel=$(\'reviewer\').value;review.wardrobe_decisao=$(\'taxonomy\').value;review.observacao_taxonomia=$(\'taxnote\').value;save();}\nfunction fillMetadata(){$(\'reviewer\').value=review.responsavel;$(\'taxonomy\').value=review.wardrobe_decisao;$(\'taxnote\').value=review.observacao_taxonomia;}\nfor(const id of [\'reviewer\',\'taxonomy\',\'taxnote\'])$(id).addEventListener(\'input\',metadata);\nfunction filterCases(keep){\n const map=answers(),group=$(\'groupfilter\').value,source=$(\'sourcefilter\').value,status=$(\'statusfilter\').value;\n visible=DATA.cases.filter(c=>(group===\'todos\'||c.grupo===group)&&(source===\'todos\'||c.fonte===source)&&(status===\'todos\'||map.get(c.case_id).status===status));\n position=Math.max(0,visible.findIndex(c=>c.case_id===keep));\n $(\'jump\').replaceChildren();visible.forEach((c,i)=>{const o=document.createElement(\'option\');o.value=String(i);o.textContent=(i+1)+\' / \'+visible.length+\' · \'+c.fonte+\' · \'+c.classe+\' · \'+c.motivo;$(\'jump\').append(o);});\n render();\n}\nfunction showCase(){if(!current)return;const offset=window.innerWidth>800?document.querySelector(\'.toolbar\').getBoundingClientRect().height+12:12;window.scrollTo({top:$(\'casebody\').getBoundingClientRect().top+window.scrollY-offset,behavior:\'instant\'});}\nfunction setAnswer(k,value){const row=answers().get(current.case_id);row[k]=value;row.status=\'pendente\';save();}\nfunction render(){\n current=visible[position]||null;$(\'casebody\').hidden=!current;\n for(const id of [\'prev\',\'next\',\'complete\',\'pending\',\'jump\'])$(id).disabled=!current;\n if(!current){message(\'Nenhum caso corresponde aos filtros.\');return;}\n $(\'jump\').value=String(position);$(\'prev\').disabled=position===0;$(\'next\').disabled=position===visible.length-1;\n const row=answers().get(current.case_id),isTrain=TRAIN.has(current.papel);\n $(\'caseheading\').textContent=current.classe+\' · \'+current.imagem;\n $(\'badges\').replaceChildren();for(const txt of [current.fonte,current.papel,current.grupo,row.status]){const e=document.createElement(\'span\');e.className=\'badge\';e.textContent=txt;$(\'badges\').append(e);}\n $(\'reason\').textContent=\'Seleção: \'+current.motivo.replaceAll(\'_\',\' \');\n $(\'context\').textContent=isTrain?\'Revisão de uma caixa de treino. A maior caixa da classe foi escolhida, sem previsões.\':\n  \'Objeto em foco — prioridade V2: \'+current.contexto.prioridade_ref+\'. Alerta no candidato 10: \'+(current.contexto.alerta_candidato10?\'sim\':\'não\')+\'; no ajuste B, época 3: \'+(current.contexto.alerta_B_e03?\'sim\':\'não\')+\'. Indícios registrados: \'+[current.contexto.indicios_candidato10,current.contexto.indicios_B_e03].filter(Boolean).join(\' | \');\n $(\'caseimage\').src=current.image_data;$(\'fields\').replaceChildren();\n for(const [key,entry] of Object.entries(DATA.fields)){\n  const wrap=document.createElement(\'div\'),label=document.createElement(\'label\'),select=document.createElement(\'select\');label.htmlFor=\'answer_\'+key;label.textContent=entry[0];select.id=\'answer_\'+key;\n  let options=[[\'pendente\',\'Selecione / pendente\'],...entry[1]];\n  if(key===\'causa\')options=isTrain?[[\'nao_aplica_treino\',\'Não se aplica: caso de treino\']]:options.filter(o=>o[0]!==\'nao_aplica_treino\');\n  if(key===\'destino_treino\')options=isTrain?options.filter(o=>o[0]!==\'nao_aplica_validacao\'):[[\'nao_aplica_validacao\',\'Fora do treino: validação\']];\n  for(const [v,text] of options){const o=document.createElement(\'option\');o.value=v;o.textContent=text;select.append(o);}\n  select.value=row[key];select.disabled=(key===\'causa\'&&isTrain)||(key===\'destino_treino\'&&!isTrain);\n  select.addEventListener(\'change\',()=>setAnswer(key,select.value));wrap.append(label,select);$(\'fields\').append(wrap);\n }\n $(\'observation\').value=row.observacao;$(\'caseid\').textContent=\'Caso \'+current.case_id+\' · SHA da imagem \'+current.image_sha256;\n progress();\n}\nfor(const id of [\'groupfilter\',\'sourcefilter\',\'statusfilter\'])$(id).addEventListener(\'change\',()=>{message(\'\');filterCases(current?.case_id);});\n$(\'observation\').addEventListener(\'input\',()=>setAnswer(\'observacao\',$(\'observation\').value));\n$(\'prev\').addEventListener(\'click\',()=>{position=Math.max(0,position-1);message(\'\');render();showCase();});\n$(\'next\').addEventListener(\'click\',()=>{position=Math.min(visible.length-1,position+1);message(\'\');render();showCase();});\n$(\'jump\').addEventListener(\'change\',()=>{position=Number($(\'jump\').value);message(\'\');render();showCase();});\n$(\'complete\').addEventListener(\'click\',()=>{\n if(!current)return;const row=answers().get(current.case_id);row.status=\'revisado\';\n try{validate(review);}catch(e){row.status=\'pendente\';message(e.message);$(\'message\').scrollIntoView({block:\'center\',behavior:\'instant\'});return;}\n const keep=visible[Math.min(position+1,visible.length-1)]?.case_id;save();message(\'Caso revisado.\');filterCases(keep);showCase();\n});\n$(\'pending\').addEventListener(\'click\',()=>{if(current){answers().get(current.case_id).status=\'pendente\';save();message(\'Caso mantido como pendente.\');render();}});\nfunction download(text,name,type){const url=URL.createObjectURL(new Blob([text],{type}));const a=document.createElement(\'a\');a.href=url;a.download=name;document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),1000);}\n$(\'export\').addEventListener(\'click\',()=>{try{metadata();validate(review);download(JSON.stringify(review,null,2),\'revisoes_13.json\',\'application/json;charset=utf-8\');message(\'JSON exportado para documentação da auditoria.\');}catch(e){message(e.message);}});\n$(\'exportcsv\').addEventListener(\'click\',()=>{try{validate(review);const keys=[\'case_id\',\'status\',...Object.keys(DATA.fields),\'observacao\'];const csv=[keys,...review.casos.map(r=>keys.map(k=>r[k]))].map(row=>row.map(v=>\'"\'+String(v??\'\').replaceAll(\'"\',\'""\')+\'"\').join(\',\')).join(\'\\r\\n\');download(\'\\ufeff\'+csv,\'revisoes_13.csv\',\'text/csv;charset=utf-8\');message(\'CSV exportado. Para importar no notebook use o JSON.\');}catch(e){message(e.message);}});\n$(\'importfile\').addEventListener(\'change\',async event=>{try{const file=event.target.files[0];if(!file)return;const imported=JSON.parse(await file.text());validate(imported);review=structuredClone(imported);fillMetadata();save();filterCases(current?.case_id);message(\'Revisões importadas para esta amostra.\');}catch(e){message(e.message);}event.target.value=\'\';});\nfunction zoom(){$(\'zoomimage\').src=current.image_data;$(\'zoom\').showModal();}\n$(\'caseimage\').addEventListener(\'click\',zoom);$(\'caseimage\').addEventListener(\'keydown\',e=>{if(e.key===\'Enter\')zoom();});$(\'closezoom\').addEventListener(\'click\',()=>$(\'zoom\').close());\nfillMetadata();filterCases();\n</script></html>\n'}

PASTA_CODIGO=PASTA_LOCAL/'codigo'
PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    (PASTA_CODIGO/nome).write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import diag11_data as dados11
import exp12_data as dados12
import audit13 as auditoria
print('Código de auditoria carregado, sem inicializar modelos.')


## 3. Conectar ao Drive e conferir os registros

Primeiro, autentica os pacotes pequenos e confere os registros do experimento
12. O candidato 10 e o checkpoint B da terceira época serão comparados
somente nos objetos registrados. Os pesos em uso permanecem preservados.


In [ ]:
from pathlib import Path
import base64
import configparser
import json
import shutil
import subprocess
import urllib.request
import zipfile
from kaggle_secrets import UserSecretsClient

SECRET_DRIVE = "GDRIVE_RCLONE_CONFIG_B64"
REMOTE_DRIVE = "gdrive_tcc"
ORIGEM_DRIVE = f"{REMOTE_DRIVE}:TCC_Visao_Computacional"
DRIVE_TCC = Path("/tmp/tcc_100k_entradas")
ENVIAR_RESULTADOS_AO_DRIVE = True
EXECUTAR_CALIBRACAO = False

# Credenciais ficam fora da pasta de resultados salva pelo Kaggle.
PASTA_RCLONE = Path("/tmp/tcc_rclone")
PASTA_RCLONE.mkdir(mode=0o700, parents=True, exist_ok=True)
PASTA_RCLONE.chmod(0o700)
CONFIG_RCLONE = PASTA_RCLONE / "gdrive_tcc.conf"
RCLONE = PASTA_RCLONE / "rclone"

try:
    segredo = UserSecretsClient().get_secret(SECRET_DRIVE)
except Exception:
    raise RuntimeError(
        f"Crie o Secret {SECRET_DRIVE} em Add-ons → Secrets e habilite "
        "o acesso deste notebook. Siga a preparação descrita acima."
    ) from None
try:
    configuracao_bytes = base64.b64decode(segredo.strip(), validate=True)
    configuracao = configparser.ConfigParser(interpolation=None)
    configuracao.read_string(configuracao_bytes.decode("utf-8-sig"))
except Exception:
    raise RuntimeError(
        "O Secret não contém a configuração rclone codificada corretamente. "
        "Copie novamente pelo comando PowerShell e atualize o Secret."
    ) from None
if (not configuracao.has_section(REMOTE_DRIVE)
        or configuracao.get(REMOTE_DRIVE, "type", fallback="") != "drive"
        or not configuracao.get(REMOTE_DRIVE, "token", fallback="")
        or not configuracao.get(REMOTE_DRIVE, "client_id", fallback="")):
    raise RuntimeError(
        "A configuração precisa da conexão gdrive_tcc, tipo drive, "
        "cliente OAuth próprio e autorização concluída no navegador."
    )
if (ENVIAR_RESULTADOS_AO_DRIVE
        and configuracao.get(REMOTE_DRIVE, "scope", fallback="") != "drive"):
    raise RuntimeError(
        "Para salvar os resultados, configure scope 1 (drive, leitura e "
        "gravação) no rclone e atualize o Secret."
    )
# Esta conexão usa Meu Drive, não um Drive compartilhado.
configuracao.remove_option(REMOTE_DRIVE, "team_drive")
with CONFIG_RCLONE.open("w", encoding="utf-8") as arquivo_config:
    configuracao.write(arquivo_config)
CONFIG_RCLONE.chmod(0o600)
del segredo, configuracao_bytes, configuracao

if not RCLONE.is_file():
    import os

    executavel = shutil.which("rclone")
    if executavel is None:
        prefixo = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(prefixo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(
            prefixo + ["apt-get", "install", "-y", "--no-install-recommends", "rclone"],
            check=True,
        )
        executavel = shutil.which("rclone")

    if executavel is None:
        raise RuntimeError("A instalação terminou, mas o executável rclone não foi localizado.")

    shutil.copy2(executavel, RCLONE)
    RCLONE.chmod(0o700)
    print("rclone instalado pelo gerenciador de pacotes do Linux.")

DRIVE_TCC.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(DRIVE_TCC).free < 10 * 1024**3:
    raise RuntimeError("Reserve pelo menos 10 GiB livres para baixar as entradas.")

def baixar_arquivo_drive(relativo, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    print("Baixando:", relativo, flush=True)
    concluido = subprocess.run(
        [str(RCLONE), "--config", str(CONFIG_RCLONE), "copyto",
         f"{ORIGEM_DRIVE}/{relativo}", str(destino),
         "--retries", "3", "--low-level-retries", "5"], check=False,
    )
    if concluido.returncode != 0 or not destino.is_file():
        raise RuntimeError(f"Falha ao baixar {relativo}. Confira a saída do rclone, a autorização e a Internet.")
    return destino


def enviar_resultados_drive(etapa="manual", obrigatorio=False):
    if not ENVIAR_RESULTADOS_AO_DRIVE:
        return False
    pasta = Path(PASTA_RESULTADOS)
    destino = f"{ORIGEM_DRIVE}/resultados/{NOME_EXPERIMENTO}"
    print(f"Enviando resultados ao Drive ({etapa})...", flush=True)
    try:
        enviado = subprocess.run(
            [str(RCLONE), "--config", str(CONFIG_RCLONE), "copy",
             str(pasta), destino, "--exclude", "**/*.tmp",
             "--retries", "2", "--low-level-retries", "3",
             "--contimeout", "15s", "--timeout", "60s"],
            check=False,
        )
        if enviado.returncode != 0:
            raise RuntimeError(f"rclone retornou código {enviado.returncode}")
    except (OSError, RuntimeError) as exc:
        mensagem = (
            f"Envio ao Drive falhou em {etapa}: {exc}. "
            "Os arquivos locais continuam em /kaggle/working/resultados_tcc. "
            "Confira a autorização e execute a última etapa para repetir o envio."
        )
        if obrigatorio:
            raise RuntimeError(mensagem) from None
        print(mensagem, flush=True)
        return False
    print("Resultados copiados ao Drive.", flush=True)
    return True

def enviar_resultados(etapa):
    return enviar_resultados_drive(etapa,obrigatorio=True)

ENTRADAS=PASTA_LOCAL/'entradas'
OI_RELATIVO='datasets/openimages_v7_tcc_100k_v1'
PACOTE_12=baixar_arquivo_drive(PASTA_EXECUCAO_12+'/pacote_experimento_12.zip',ENTRADAS/'pacote_12.zip')
PACOTE_11=baixar_arquivo_drive(PASTA_EXECUCAO_11+'/pacote_diagnostico_11.zip',ENTRADAS/'pacote_11.zip')
PREP=baixar_arquivo_drive(OI_RELATIVO+'/configuracao_preparacao.json',ENTRADAS/'preparacao.json')
PACOTES=baixar_arquivo_drive(OI_RELATIVO+'/manifesto_pacotes.json',ENTRADAS/'manifesto_pacotes.json')
DIVISAO_CSV=baixar_arquivo_drive(PASTA_EXECUCAO_10+'/divisao_openimages.csv',ENTRADAS/'divisao.csv')
DIVISAO_JSON=baixar_arquivo_drive(PASTA_EXECUCAO_10+'/divisao_openimages.json',ENTRADAS/'divisao.json')
BUNDLE=auditoria.load_source12(PACOTE_12,EXPECTED['pacote_12'],EXPECTED['codigo_12'])
if (BUNDLE['config']['expected']['candidato']!=EXPECTED['candidato'] or
    BUNDLE['summary']['dataset_sha256']!=EXPECTED['dataset_novo']):
    raise RuntimeError('Identidades das referências ou do dataset foram alteradas.')
RESUMO_11,CONFIG_11,LOCAL_11,MANIFESTO_NOVO=dados12.read_package11(
    PACOTE_11,EXPECTED['pacote_11'],{'antigo':EXPECTED['baseline'],'candidato':EXPECTED['candidato']},EXPECTED['dataset_novo'])
if LOCAL_11!=LOCALIZACAO_11:raise RuntimeError('Localização da validação diferente da original.')
preparacao=json.loads(PREP.read_text())
if hashlib.sha256(core.canonical(preparacao).encode()).hexdigest()!=EXPECTED['preparacao']:
    raise RuntimeError('Preparação ou mapeamento de classes alterados.')
if core.sha256_file(PACOTES)!=EXPECTED['manifesto_pacotes']:
    raise RuntimeError('Manifesto dos lotes alterado.')
HISTORICO=dados11.load_history(DIVISAO_CSV,DIVISAO_JSON,EXPECTED['divisao_10'])
PLANO_ERROS,RECIBOS_ERROS=auditoria.error_plan(BUNDLE,SEED)
PLANO_OI=auditoria.replay_plan(BUNDLE['replay'],HISTORICO,SEED)
core.save_json(PASTA_RESULTADOS/'selecao_erros.json',RECIBOS_ERROS)
core.save_csv(PASTA_RESULTADOS/'diferencas_por_classe.csv',auditoria.class_differences(BUNDLE))
core.save_json(PASTA_RESULTADOS/'referencia_experimento_12.json',BUNDLE['summary'])
print('Objetos de validação selecionados:',len(PLANO_ERROS))
print('Imagens Open Images TRAIN a recuperar:',len(PLANO_OI))
enviar_resultados('registros e seleção dos casos')


## 4. Recuperar imagens e labels originais, com a reserva protegida

Home TRAIN/VAL são separados pelos papéis originais. A validação externa
é exatamente a do notebook 11. O extrator não abre nem escreve os JPEGs,
labels ou coberturas das 300 imagens reservadas. Somente seu manifesto
é usado para bloquear coincidências com dados de treino.

Os casos Open Images de treino vêm dos 2.000 IDs do replay autenticado do
experimento 12. São 24 com classe primária wardrobe, seis table e seis chair.
Nada vem do TEST histórico de 20 mil imagens.


In [ ]:
ZIP_HOME=baixar_arquivo_drive('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip')
ZIP_NOVO=baixar_arquivo_drive(LOCALIZACAO_11['pasta_drive']+'/'+LOCALIZACAO_11['arquivo'],ENTRADAS/'validacao_11.zip')
HOME,HASHES_HOME,PHASH_HOME,AUDITORIA_HOME=dados11.extract_home(
    ZIP_HOME,PASTA_LOCAL/'home',EXPECTED['home_zip'],EXPECTED['test_home'],EXPECTED['politica'])
VAL_NOVA,IDS_RESERVA,FROZEN_FILES=dados12.load_frozen_validation(
    ZIP_NOVO,PASTA_LOCAL/'validacao',EXPECTED['zip_novo'],EXPECTED['dataset_novo'],MANIFESTO_NOVO)
if len(IDS_RESERVA)!=300 or any('reserva_futura' in p.parts for p in (PASTA_LOCAL/'validacao').rglob('*')):
    raise RuntimeError('A reserva não foi mantida isolada.')
novos=MANIFESTO_NOVO['arquivos']
if ({r['id'] for r in novos}&HISTORICO['ids'] or
    {r['sha256'] for r in novos}&(HISTORICO['hashes']|HASHES_HOME)):
    raise RuntimeError('Coincidência entre dados externos e o histórico.')
proibidos_hash=HASHES_HOME|{r['sha256'] for r in novos}
indice=dados11.NearIndex()
for r in HOME:
    if r['role']=='home_val':indice.add(r['phash'],r['id'])
for r in novos:indice.add(int(r['phash'],16),r['id'])
pacotes=json.loads(PACOTES.read_text())
if pacotes['config_sha256']!=EXPECTED['preparacao']:raise RuntimeError('Configuração dos lotes incompatível.')
nomes_lotes={r['lote'] for r in PLANO_OI}
lotes=sorted([r for r in pacotes['lotes'] if r['arquivo'] in nomes_lotes],key=lambda r:r['arquivo'])
if len(lotes)!=len(nomes_lotes) or not nomes_lotes<={f'lote_{i:06}.zip' for i in range(1,6)}:
    raise RuntimeError('Lotes fora do piloto histórico.')
def buscar_lote(item,destino):
    return baixar_arquivo_drive(OI_RELATIVO+'/lotes/'+item['arquivo'],destino)
def validar_label(raw,nclasses=12,source='label'):
    return [o['classe_id'] for o in core.parse_yolo(raw,1,1,source=source)]
ids_test={r['id'] for r in HISTORICO['rows'] if r['split']=='test'}
REPLAY,EXCLUSOES,congelados,RECIBOS_LOTES=dados12.load_replay(
    lotes,PLANO_OI,PASTA_LOCAL/'openimages_train',buscar_lote,EXPECTED['preparacao'],
    ids_test|{r['id'] for r in novos},proibidos_hash,indice,validar_label,
    final_size=len(PLANO_OI),seed=SEED)
if EXCLUSOES:raise RuntimeError('Um caso do replay antigo foi excluído por semelhança; verificar antes de mudar a amostra.')
FROZEN_FILES+=congelados
FROZEN_FILES += [(p,core.sha256_file(p)) for p in [PACOTE_12,PACOTE_11,PREP,PACOTES,DIVISAO_CSV,DIVISAO_JSON,ZIP_HOME,ZIP_NOVO]]
FROZEN_FILES += [(r['image'],r['sha256']) for r in HOME]
print('Dados autenticados: Home TRAIN/VAL, externa VAL e',len(REPLAY),'imagens OI TRAIN.')
print('Nenhuma imagem de TEST ou reserva foi recuperada para revisão.')


## 5. Gerar os casos e o revisor visual

Em treino, a seleção usa hash e presença da classe; a maior caixa da classe
fica em foco. Em validação, prioriza regressões de mesas e cadeiras, incluindo
as 14 novas omissões de prioridade alta observadas nesses objetos na externa.
Também inclui omissões persistentes, melhoras e controles.

Os painéis de previsões mostram **somente as classes avaliáveis que constam
nos CSVs a partir de confiança 0,25**. O pacote 12 não contém todos os caches
de previsões. Assim, esta reconstrução não mostra classes sem cobertura ou
detecções abaixo desse limiar. Indícios abaixo do limiar permanecem descritos
no texto quando estavam registrados no CSV.


In [ ]:
CASOS,REGISTROS=auditoria.build_cases(HOME,REPLAY,VAL_NOVA,PLANO_ERROS,SEED)
INPUTS={'pacote12_sha256':EXPECTED['pacote_12'],'pacote11_sha256':EXPECTED['pacote_11'],
    'home_zip_sha256':EXPECTED['home_zip'],'dataset_externo_sha256':EXPECTED['dataset_novo'],
    'divisao80_20_sha256':EXPECTED['divisao_10'],'preparacao_sha256':EXPECTED['preparacao']}
MANIFESTO=auditoria.make_manifest(CASOS,INPUTS,RECIBOS_ERROS,SEED)
core.save_json(PASTA_RESULTADOS/'manifesto_auditoria_13.json',MANIFESTO)
core.save_csv(PASTA_RESULTADOS/'indice_casos.csv',[
    {k:c[k] for k in ['case_id','id','fonte','papel','grupo','classe','motivo','arquivo','image_sha256','label_sha256']}
    for c in CASOS])
auditoria.render_cases(CASOS,REGISTROS,BUNDLE,PASTA_RESULTADOS)
REVISOES=auditoria.empty_review(MANIFESTO)
if ARQUIVO_REVISOES is not None:
    REVISOES=json.loads(Path(ARQUIVO_REVISOES).read_text(encoding='utf-8-sig'))
STATUS=auditoria.write_review_outputs(MANIFESTO,REVISOES,PASTA_RESULTADOS)
REVISOR=auditoria.build_reviewer(MANIFESTO,REVISOES,MODULOS_EMBUTIDOS['revisor_template.html'],PASTA_RESULTADOS)
print('Casos por grupo:',dict(Counter(c['grupo'] for c in CASOS)))
print('Casos revisados:',STATUS['casos_revisados'],'| pendentes:',STATUS['casos_pendentes'])
print('Amostra congelada:',MANIFESTO['audit_manifest_sha256'])
enviar_resultados('galerias e formulário de revisão')


## 6. Conferir integridade e salvar o pacote

O resumo separa a geração das fichas da conclusão da revisão humana. Na
primeira execução, as fichas ficam pendentes. O arquivo de triagem contém
somente casos de treino e não é um dataset pronto: revisar uma caixa não
comprova que todas as anotações da imagem estejam corretas.

As respostas nunca alteram os labels originais. Ampliar wardrobe para outros
móveis ou separar classes exige reanotação documentada e nova versão do
mapeamento e da cobertura, antes de outra comparação. As validações históricas
permanecem preservadas, e a reserva continua fora de seleção e revisão.


In [ ]:
dados12.verify_frozen(FROZEN_FILES)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    path=PASTA_RESULTADOS/'codigo'/nome;path.parent.mkdir(exist_ok=True);path.write_text(fonte,encoding='utf-8')
core.save_json(PASTA_RESULTADOS/'configuracao_auditoria_13.json',{
    'versao':auditoria.VERSION,'experimento':NOME_EXPERIMENTO,'seed':SEED,
    'inputs':INPUTS,'audit_manifest_sha256':MANIFESTO['audit_manifest_sha256'],
    'codigo_sha256':{n:hashlib.sha256(s.encode()).hexdigest() for n,s in MODULOS_EMBUTIDOS.items()},
    'ambiente':{'python':platform.python_version(),'numpy':np.__version__,'pillow':PIL.__version__,'scipy':scipy.__version__},
    'ambiente_execucao':'CPU; sem Torch/Ultralytics no código executado',
    'novos_treinamentos':0,'inferencias_novas':0,'reserva_futura_extraida':False,'reserva_futura_inferida':False,
    'integridade_conferida':True,'home':AUDITORIA_HOME,'proveniencia_baseline':RESUMO_11['auditoria']['baseline_original'],
    'lotes_conferidos':RECIBOS_LOTES,'mids_raizes_originais':preparacao['mids_raizes']})
core.save_json(PASTA_RESULTADOS/'criterios_recorte_futuro.json',{
    'status':'proposta; não é um dataset pronto','somente_papeis':['home_train','train'],
    'cenas':'fotografias internas revisadas','taxonomia_wardrobe':REVISOES['wardrobe_decisao'],
    'requer_revisao_de_todos_os_rotulos_da_imagem':True,'correcoes_aplicadas':False,
    'validacao_nunca_vira_treino':True,'reserva_preservada':True,
    'proximo_treino_executado':False})
(PASTA_RESULTADOS/'LEIA_PRIMEIRO.md').write_text(
    '# Auditoria 13\n\nAbra revisor_13.html no navegador após extrair o ZIP. Não precisa de Internet. '
    'As respostas podem ser exportadas em revisoes_13.json. O pacote documenta a auditoria; o JSON exportado registra as revisões preenchidas.\n\n'
    'A geração terminou; a revisão humana pode continuar pendente. A amostra é dirigida, com uma caixa em foco por caso. '
    'Nenhum label foi alterado, nenhum modelo foi executado e as 300 reservadas não foram extraídas. '
    'Imagens de validação nunca entram na triagem de treino. A triagem não constitui um dataset final.\n',encoding='utf-8')
PACOTE_FINAL=auditoria.package_reports(PASTA_RESULTADOS)
enviar_resultados('pacote final da auditoria')
print('Auditoria preparada:',len(CASOS),'casos; revisão humana concluída:',STATUS['revisao_humana_concluida'])
print('Arquivo de resultados:',PACOTE_FINAL)
print('Para registrar decisões, abra revisor_13.html e baixe o JSON das revisões.')
display(FileLink(str(PACOTE_FINAL)))
display(FileLink(str(REVISOR)))


## 7. O que enviar e como continuar

- Resultados em **pacote_auditoria_13.zip** para analisarmos as galerias e os registros.
- Se o grupo revisar as fichas, extraia o ZIP e abra **revisor_13.html**.
  Preencha as respostas, proponha o significado de wardrobe e use
  **Baixar revisões em JSON**. Resultados em também **revisoes_13.json**.
- O formulário guarda respostas no navegador quando possível; o JSON
  exportado é a forma de preservar e compartilhar a revisão. O CSV é opcional.
- Não é preciso iniciar outro treinamento. O passo seguinte depende de
  separar erros de rótulo, de caixa e de detecção e definir uma taxonomia
  coerente para um recorte futuro de treino. A reserva não entra nessa decisão.
